# Astrometry v11: head vs raw, and are the worsened sources a real population?

Fresh, from-scratch look on the **v11** stack (foundation `jaisp_v11_q1_soft` + plain retrained
position head; archive `latent_position_v11_q1_plain/anchors_centernet_v11plain.npz`, all matched
anchors, no dedup, 790 ECDFS tiles, 9 bands).

1. **Plot set 1** (as nb18): per-source raw cross-band offset vs offset after the head, colored by S/N.
2. **Plot set 2** (as nb18): 4-panel 2D histogram of the same plane — density, median VIS mag, median S/N, median size.
3. **New question**: the sources the head makes *worse* (residual > raw) — are they a random draw
   (unlucky noise realizations, raw happened to land close) or a coherent population the model fails
   on for a reason? Tests: cross-band coherence vs a chance null, matched-control property
   comparison, head-confidence (sigma) check, spatial and direction coherence.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

def find_repo_root(s=None):
    s = (s or Path.cwd()).resolve()
    for c in [s, *s.parents]:
        if (c/'models').exists() and (c/'data').exists(): return c
    raise FileNotFoundError
REPO = find_repo_root()
OUTDIR = REPO/'io/_nb31_outputs'; OUTDIR.mkdir(exist_ok=True)
ARCH = REPO/'models/checkpoints/latent_position_v11_q1_plain/anchors_centernet_v11plain.npz'
d = np.load(ARCH, allow_pickle=True)
BANDS = ['u','g','r','i','z','y','nisp_Y','nisp_J','nisp_H']

# flat per-measurement arrays (one row = one anchor in one band)
RAWV=[]; RESV=[]; SNR=[]; SIG=[]; RA=[]; DEC=[]; TILE=[]; BAND=[]
for bi,b in enumerate(BANDS):
    rr = np.asarray(d[f'{b}_raw'])*1000.       # mas, vector
    hr = np.asarray(d[f'{b}_head_resid'])*1000.
    RAWV.append(rr); RESV.append(hr)
    SNR.append(np.asarray(d[f'{b}_snr'])); SIG.append(np.asarray(d[f'{b}_sigma'])*1000.)
    RA.append(np.asarray(d[f'{b}_ra'])); DEC.append(np.asarray(d[f'{b}_dec']))
    TILE.append(np.asarray(d[f'{b}_tiles'])); BAND.append(np.full(len(rr), bi))
RAWV=np.vstack(RAWV); RESV=np.vstack(RESV)
SNR=np.concatenate(SNR); SIG=np.concatenate(SIG)
RA=np.concatenate(RA); DEC=np.concatenate(DEC)
TILE=np.concatenate(TILE); BAND=np.concatenate(BAND)
RAW=np.hypot(*RAWV.T); RES=np.hypot(*RESV.T)
ok=(RAW>0)&(RES>0)&np.isfinite(RAW)&np.isfinite(RES)&np.isfinite(SNR)&(SNR>0)
RAWV,RESV,RAW,RES,SNR,SIG,RA,DEC,TILE,BAND=[a[ok] for a in (RAWV,RESV,RAW,RES,SNR,SIG,RA,DEC,TILE,BAND)]
print(f'N band-measurements: {len(RAW):,}   (bands: {len(BANDS)}, tiles: {len(np.unique(TILE))})')
print(f'raw median {np.median(RAW):.1f} mas -> head median {np.median(RES):.1f} mas | '
      f'improved (res<raw): {(RES<RAW).mean():.1%}')

# ---- plot set 1: scatter raw vs head residual, colored by S/N ----
rng = np.random.default_rng(0)
idx = rng.choice(len(RAW), min(30000,len(RAW)), replace=False)
o = np.argsort(SNR[idx])
fig,ax = plt.subplots(figsize=(7.5,7))
sc = ax.scatter(RAW[idx][o], RES[idx][o], c=SNR[idx][o], s=4, alpha=0.35, edgecolors='none',
                cmap='viridis', norm=LogNorm(vmin=5, vmax=100), rasterized=True)
cb = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.02); cb.set_label('S/N', fontsize=12)
lim=[0.4,300]
ax.plot(lim,lim,'k--',lw=1,alpha=0.7,label='y = x (no change)')
ax.axhline(np.median(RES),color='#c84b4b',ls=':',lw=1.2,label=f'head median {np.median(RES):.0f} mas')
ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect('equal')
ax.set_xlabel('Raw offset [mas]',fontsize=13); ax.set_ylabel('Residual (head) offset [mas]',fontsize=13)
ax.grid(True,which='both',alpha=0.2); ax.legend(fontsize=10,loc='lower right')
ax.set_title(f'v11 foundation + plain head — all bands, 790 tiles  (N={len(RAW):,})',fontsize=11)
fig.savefig(OUTDIR/'nb31_raw_vs_resid_scatter.png',dpi=130,bbox_inches='tight')
print('saved', (OUTDIR/'nb31_raw_vs_resid_scatter.png').relative_to(REPO)); plt.show()

In [ ]:
# ---- plot set 2: 4-panel 2D-histogram view [density, VIS mag, S/N, size] ----
# Square geometric bins on log-log axes; ALL panels hide bins with < Nmin sources.
import matplotlib.patheffects as pe
from astropy.io import fits
from scipy.spatial import cKDTree
from scipy.stats import binned_statistic_2d

cosd = np.cos(np.deg2rad(np.median(DEC)))
def match_cat(cra, cdec, vals, rad=0.5):
    t = cKDTree(np.column_stack([cra*cosd, cdec]))
    dist, idx = t.query(np.column_stack([RA*cosd, DEC]), k=1)
    return np.where(dist*3600 < rad, vals[idx], np.nan)

mc = fits.open(REPO/'data/edf_s_ood/catalogs_compact/mer_FINAL_q1_ECDFS_footprint.fits')[1].data
vmag = match_cat(np.asarray(mc['ra'],float), np.asarray(mc['dec'],float), np.asarray(mc['mag_vis'],float))
hs = fits.open(REPO/'data/edf_s_ood/catalogs_compact/mer_q1_ECDFS_Hsize.fits')[1].data
size = match_cat(np.asarray(hs['ra'],float), np.asarray(hs['dec'],float),
                 np.asarray(hs['semimajor_axis'],float)*0.1)

LO,HI=0.5,250; NB=55; Nmin=10; VIS_PIX=100.0; CMAP='cividis'
edges = np.geomspace(LO,HI,NB+1)
cnt_all = binned_statistic_2d(RAW,RES,None,'count',bins=[edges,edges]).statistic
cnt_all = np.ma.masked_where(cnt_all<Nmin, cnt_all)
def grid_median(C):
    mm = np.isfinite(C)
    med = binned_statistic_2d(RAW[mm],RES[mm],C[mm],'median',bins=[edges,edges]).statistic
    cnt = binned_statistic_2d(RAW[mm],RES[mm],None,'count',bins=[edges,edges]).statistic
    return np.ma.masked_where(cnt<Nmin, med)
def cbar(fig,im,ax,label):
    cb = fig.colorbar(im,ax=ax,fraction=0.046,pad=0.02)
    cb.ax.text(0.5,0.5,label,transform=cb.ax.transAxes,rotation=90,ha='center',va='center',
               fontsize=11,color='white',
               path_effects=[pe.withStroke(linewidth=2.2,foreground='black',alpha=0.6)])

frac_improved=(RES<RAW).mean(); raw_med=np.median(RAW); res_med=np.median(RES)
fig,axes = plt.subplots(2,2,figsize=(14,12.5)); axes=axes.ravel()
def setup(ax):
    ax.plot([LO,HI],[LO,HI],'k--',lw=1,alpha=0.6,zorder=5)
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlim(LO,HI); ax.set_ylim(LO,HI); ax.set_aspect('equal')
    ax.set_xlabel('Raw offset [mas]'); ax.set_ylabel('Residual (head) offset [mas]')

im = axes[0].pcolormesh(edges,edges,cnt_all.T,cmap=CMAP,norm=LogNorm(),shading='flat')
setup(axes[0]); cbar(fig,im,axes[0],'source count (log)')
axes[0].axvline(VIS_PIX,color='darkorange',ls='--',lw=1.4,zorder=6)
axes[0].axhline(VIS_PIX,color='darkorange',ls='--',lw=1.4,zorder=6)
axes[0].text(VIS_PIX*1.06,LO*1.4,f'VIS pixel ({VIS_PIX:.0f} mas)',color='darkorange',fontsize=9,
             rotation=90,ha='left',va='bottom',fontweight='bold')
axes[0].text(0.04,0.96,f'{frac_improved:.0%} below 1:1  (head improves)\n'
             f'median {raw_med:.0f} $\\rightarrow$ {res_med:.0f} mas,  N={len(RAW):,}',
             transform=axes[0].transAxes,va='top',ha='left',fontsize=10.5,
             bbox=dict(boxstyle='round',fc='white',ec='0.6',alpha=0.9))
for ax,(C,lab) in zip(axes[1:], [(vmag,'median VIS mag'),(SNR,'median VIS S/N'),
                                 (size,'median size (semimajor) [arcsec]')]):
    med = grid_median(C)
    vmin,vmax = np.nanpercentile(med.compressed(),[2,98])
    im = ax.pcolormesh(edges,edges,med.T,cmap=CMAP,vmin=vmin,vmax=vmax,shading='flat')
    setup(ax); cbar(fig,im,ax,lab)
    ax.text(0.04,0.96,f'bins < {Nmin} src hidden',transform=ax.transAxes,va='top',ha='left',
            fontsize=9,color='0.3',bbox=dict(boxstyle='round',fc='white',ec='none',alpha=0.7))
plt.tight_layout()
fig.savefig(OUTDIR/'nb31_hist2d_4panel.png',dpi=130,bbox_inches='tight')
print('saved', (OUTDIR/'nb31_hist2d_4panel.png').relative_to(REPO)); plt.show()

## Are the worsened sources a population, or just unlucky?

"Worsened" is defined two ways, tested in parallel:
- **W1 (any)**: residual > raw. Includes trivial cases where raw happened to land a few mas from
  the VIS convention and any head noise pushes above it.
- **W2 (material)**: residual > raw + 10 mas **and** residual > 20 mas. The head placed the source
  materially off, not just noise around a lucky raw.

**Chance null**: each band-measurement worsens independently with the *empirical* rate for its
(band, S/N) cell. If worsening were luck, whether a source worsens in band g says nothing about
band H. So: group measurements into unique physical sources (0.3" linking), count worsened bands
k out of n matched, and compare the observed k-distribution to the null (within-(band,S/N-bin)
permutations, which preserve every marginal rate by construction). Cross-band coherence in excess
of the null = the *source* carries the failure, i.e. a population with a reason.

In [ ]:
# ---- group band-measurements into unique sources (connected components @ 0.3") ----
# The archive is NO-dedup: overlapping tiles repeat the same source, so a group can hold
# several copies of the same band. For the coherence test we dedup to ONE measurement per
# (source, band) — tile duplicates share the same coadd pixels, so they are correlated even
# under the chance null and would fake cross-band coherence.
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

XY = np.column_stack([RA*cosd, DEC])
tree = cKDTree(XY)
pairs = tree.query_pairs(0.3/3600., output_type='ndarray')
n = len(RA)
adj = coo_matrix((np.ones(len(pairs)), (pairs[:,0], pairs[:,1])), shape=(n,n))
ncomp, gid = connected_components(adj, directed=False)
print(f'{n:,} band-measurements -> {ncomp:,} unique sources')

W1 = RES > RAW
W2 = (RES > RAW + 10.) & (RES > 20.)

# side check on the tile duplicates before discarding them: same source, same band, two tile
# crops — does worsening repeat? (shares the coadd pixels, so this only rules out crop/model
# stochasticity, not a lucky raw; the band-level test below is the real one)
key = gid.astype(np.int64)*16 + BAND
uniqk, firstk, cntk = np.unique(np.sort(key), return_index=True, return_counts=True)
ordk = np.argsort(key, kind='stable')
repk = cntk >= 2
i1 = ordk[firstk[repk]]; i2 = ordk[firstk[repk]+1]
for W,tag in [(W1,'W1 (res>raw)'),(W2,'W2 (material)')]:
    p1, p2, p11 = W[i1].mean(), W[i2].mean(), (W[i1]&W[i2]).mean()
    print(f'tile-repeat {tag:15s}: P(worsen in both crops)={p11:.3f} vs independent {p1*p2:.3f} '
          f'-> lift x{p11/max(p1*p2,1e-9):.1f}  (N pairs={repk.sum():,})')

# dedup: keep the highest-S/N copy per (source, band)
sel = np.lexsort((-SNR, BAND, gid))
kk = key[sel]
firstmask = np.ones(len(sel), bool); firstmask[1:] = kk[1:] != kk[:-1]
keep = sel[firstmask]
RAWV,RESV,RAW,RES,SNR,SIG,RA,DEC,TILE,BAND,gid,W1,W2 = [a[keep] for a in
    (RAWV,RESV,RAW,RES,SNR,SIG,RA,DEC,TILE,BAND,gid,W1,W2)]
print(f'after (source, band) dedup: {len(RAW):,} measurements')

nb_src = np.bincount(gid, minlength=ncomp)                       # distinct bands per source (<=9)
k1 = np.bincount(gid, weights=W1, minlength=ncomp).astype(int)
k2 = np.bincount(gid, weights=W2, minlength=ncomp).astype(int)
src_ra  = np.bincount(gid, weights=RA,  minlength=ncomp)/nb_src
src_dec = np.bincount(gid, weights=DEC, minlength=ncomp)/nb_src
src_snr = np.bincount(gid, weights=SNR, minlength=ncomp)/nb_src  # mean S/N across bands
print(f'W1 rate (per measurement): {W1.mean():.1%}   W2 rate: {W2.mean():.1%}')
mb = np.bincount(nb_src)
print('sources by n distinct bands:', {i:int(c) for i,c in enumerate(mb) if c>0 and i>0})
print('per-band W2 rate:', {b: f'{W2[BAND==bi].mean():.1%}' for bi,b in enumerate(BANDS)})

# null worsening prob per measurement = empirical rate in its (band, S/N-decile) cell
snr_edges = np.quantile(SNR, np.linspace(0,1,11)); snr_edges[0]-=1; snr_edges[-1]+=1
cell = BAND*10 + np.clip(np.digitize(SNR, snr_edges)-1, 0, 9)
def null_sim(W, nsims=300, seed=1):
    p = np.bincount(cell, weights=W)/np.maximum(np.bincount(cell),1)
    pi = p[cell]
    r = np.random.default_rng(seed)
    # counts[n][k] accumulated over sims
    maxn = nb_src.max()
    out = np.zeros((nsims, maxn+1, maxn+1))
    for s in range(nsims):
        w = r.random(len(pi)) < pi
        ks = np.bincount(gid, weights=w, minlength=ncomp).astype(int)
        for nn in range(2, maxn+1):
            m = nb_src == nn
            out[s, nn] = np.bincount(ks[m], minlength=maxn+1)
    return out
sims1 = null_sim(W1); sims2 = null_sim(W2, seed=2)

fig,axs = plt.subplots(1,2,figsize=(14,5))
for ax,(W,ks,sims,tag) in zip(axs,[(W1,k1,sims1,'W1: res > raw'),
                                   (W2,k2,sims2,'W2: material (res > raw+10, >20 mas)')]):
    # pool sources with n>=4 matched bands; plot fraction with k worsened vs null
    m = nb_src >= 4
    kk = ks[m]; nn_tot = m.sum()
    obs = np.bincount(kk, minlength=10)[:10]/nn_tot
    null_counts = sims[:, 4:, :10].sum(axis=1)   # sum over n>=4
    null_frac = null_counts/nn_tot
    lo,md_,hi = np.percentile(null_frac,[2.5,50,97.5],axis=0)
    x = np.arange(10)
    ax.bar(x-0.2, obs, width=0.4, label='observed', color='#c84b4b')
    ax.bar(x+0.2, md_, width=0.4, yerr=[md_-lo,hi-md_], label='chance null (95%)', color='#3b6fb6', alpha=0.8)
    ax.set_yscale('log'); ax.set_ylim(max(obs[obs>0].min(),1e-6)/3, 1.2)
    ax.set_xlabel('k = number of bands worsened (sources with $\\geq$4 bands)')
    ax.set_ylabel('fraction of sources'); ax.set_title(tag); ax.legend()
    for kthr in [3,4,5]:
        o = (kk>=kthr).sum(); nl = null_counts[:, kthr:].sum(axis=1)
        print(f'{tag:45s} k>={kthr}: observed {o:6d}  null {nl.mean():8.1f} +- {nl.std():6.1f}  '
              f'excess x{o/max(nl.mean(),1e-9):5.1f}')
plt.tight_layout()
fig.savefig(OUTDIR/'nb31_coherence_null.png',dpi=130,bbox_inches='tight')
print('saved', (OUTDIR/'nb31_coherence_null.png').relative_to(REPO)); plt.show()

### Coherent subset and matched controls

Whatever excess the null test shows defines the **coherent subset**: sources with ≥ 4 matched bands
and k ≥ 3 *material* (W2) worsenings. Controls are drawn from sources with the same number of
matched bands and the closest mean S/N, with **zero** material worsenings, so any property
difference is a property of the failure population, not of the S/N slice it lives in.

In [ ]:
# ---- coherent subset vs S/N-matched controls: what are these sources? ----
SUB = (nb_src>=4) & (k2>=3)          # coherent material worseners
CTL_POOL = (nb_src>=4) & (k2==0)
print(f'coherent subset: {SUB.sum():,} sources   control pool: {CTL_POOL.sum():,}')

# nearest-in-logS/N control matching (with replacement-free greedy on sorted arrays)
sub_i = np.flatnonzero(SUB); pool_i = np.flatnonzero(CTL_POOL)
order = np.argsort(np.log(src_snr[pool_i])); pool_s = pool_i[order]
pos = np.searchsorted(np.log(src_snr[pool_s]), np.log(src_snr[sub_i]))
ctl_i = pool_s[np.clip(pos, 0, len(pool_s)-1)]
print(f'S/N match check: subset median {np.median(src_snr[sub_i]):.1f}, controls {np.median(src_snr[ctl_i]):.1f}')

# per-source properties from MER (match unique-source coords)
def match_src(cra, cdec, vals, rad=0.5):
    t = cKDTree(np.column_stack([cra*cosd, cdec]))
    dist, idx = t.query(np.column_stack([src_ra*cosd, src_dec]), k=1)
    return np.where(dist*3600 < rad, vals[idx], np.nan)
s_vmag = match_src(np.asarray(mc['ra'],float), np.asarray(mc['dec'],float), np.asarray(mc['mag_vis'],float))
s_plike= match_src(np.asarray(mc['ra'],float), np.asarray(mc['dec'],float),
                   (np.asarray(mc['point_like_flag'])==1).astype(float))   # 1=point-like, 999999=fill
s_size = match_src(np.asarray(hs['ra'],float), np.asarray(hs['dec'],float), np.asarray(hs['semimajor_axis'],float)*0.1)
s_pprob= match_src(np.asarray(hs['ra'],float), np.asarray(hs['dec'],float), np.asarray(hs['point_like_prob'],float))
# crowding: distance to nearest OTHER MER object
mt = cKDTree(np.column_stack([np.asarray(mc['ra'],float)*cosd, np.asarray(mc['dec'],float)]))
dd,_ = mt.query(np.column_stack([src_ra*cosd, src_dec]), k=2)
s_nnd = dd[:,1]*3600.
# head confidence: mean predicted sigma across a source's bands
s_sig = np.bincount(gid, weights=SIG, minlength=ncomp)/nb_src

fig,axs = plt.subplots(2,3,figsize=(16,9))
panels = [(s_vmag,'VIS mag',np.linspace(16,26,40)),
          (s_size,'MER semimajor axis [arcsec]',np.geomspace(0.05,3,40)),
          (src_snr,'mean S/N (matched by construction)',np.geomspace(3,3000,40)),
          (s_nnd,'nearest MER neighbor [arcsec]',np.geomspace(0.3,30,40)),
          (s_sig,'head predicted sigma [mas]',np.geomspace(3,300,40)),
          (s_pprob,'MER point-like prob',np.linspace(0,1,30))]
for ax,(v,lab,bins) in zip(axs.ravel(), panels):
    for ii,name,col in [(sub_i,'worseners (k$\\geq$3 material)','#c84b4b'),
                        (ctl_i,'S/N-matched controls','#3b6fb6')]:
        vv = v[ii]; vv = vv[np.isfinite(vv)]
        ax.hist(vv, bins=bins, density=True, histtype='step', lw=2, color=col, label=name)
        ax.axvline(np.median(vv), color=col, ls=':', lw=1)
    if bins[0]>0 and bins[1]/bins[0] != bins[2]-bins[1]:
        if not np.allclose(np.diff(bins), bins[1]-bins[0]): ax.set_xscale('log')
    ax.set_xlabel(lab); ax.legend(fontsize=8)
frac_pl_sub = np.nanmean(s_plike[sub_i]); frac_pl_ctl = np.nanmean(s_plike[ctl_i])
print(f'point_like_flag fraction: worseners {frac_pl_sub:.2f}  controls {frac_pl_ctl:.2f}')
for r_ in (1.0, 1.5, 2.0):
    print(f'close MER companion < {r_:.1f}": worseners {(s_nnd[sub_i]<r_).mean():.1%}  '
          f'controls {(s_nnd[ctl_i]<r_).mean():.1%}')
for v,lab in [(s_vmag,'VIS mag'),(s_size,'size'),(s_nnd,'NN dist'),(s_sig,'head sigma'),(s_pprob,'p(point)')]:
    a,b = np.nanmedian(v[sub_i]), np.nanmedian(v[ctl_i])
    print(f'{lab:12s} median: worseners {a:8.2f}   controls {b:8.2f}')
plt.tight_layout()
fig.savefig(OUTDIR/'nb31_population_props.png',dpi=130,bbox_inches='tight')
print('saved', (OUTDIR/'nb31_population_props.png').relative_to(REPO)); plt.show()

In [ ]:
# ---- spatial + direction coherence of the coherent worseners ----
from scipy.stats import chi2

# mean head-residual vector per source over its MATERIALLY worsened bands
sw = np.zeros((ncomp,2)); cw = np.zeros(ncomp)
np.add.at(sw, gid[W2], RESV[W2]); np.add.at(cw, gid, W2.astype(float))
mean_vec = np.full((ncomp,2), np.nan)
has = cw>0; mean_vec[has] = sw[has]/cw[has,None]

fig,axs = plt.subplots(1,3,figsize=(17,5.2))
# (a) sky map
axs[0].scatter(src_ra[ctl_i], src_dec[ctl_i], s=2, alpha=0.25, color='#3b6fb6', label='controls')
axs[0].scatter(src_ra[sub_i], src_dec[sub_i], s=6, alpha=0.6, color='#c84b4b', label='worseners')
axs[0].set_xlabel('RA [deg]'); axs[0].set_ylabel('Dec [deg]'); axs[0].invert_xaxis()
axs[0].legend(); axs[0].set_title('sky distribution')

# (b) direction of the mean residual vector, worseners vs controls with any W1 band
ang_sub = np.arctan2(mean_vec[sub_i,1], mean_vec[sub_i,0])
ctl_has = ctl_i[np.isfinite(mean_vec[ctl_i,0])]
# for controls use mean W1 residual vector instead (they have no W2 bands by construction)
sw1 = np.zeros((ncomp,2)); cw1 = np.zeros(ncomp)
np.add.at(sw1, gid[W1], RESV[W1]); np.add.at(cw1, gid, W1.astype(float))
mv1 = np.full((ncomp,2), np.nan); h1 = cw1>0; mv1[h1] = sw1[h1]/cw1[h1,None]
ang_ctl = np.arctan2(mv1[ctl_i,1], mv1[ctl_i,0]); ang_ctl = ang_ctl[np.isfinite(ang_ctl)]
for ang,name,col in [(ang_sub,'worseners','#c84b4b'),(ang_ctl,'controls (W1 vec)','#3b6fb6')]:
    ang = ang[np.isfinite(ang)]
    axs[1].hist(np.degrees(ang), bins=36, density=True, histtype='step', lw=2, color=col, label=name)
    R = np.hypot(np.mean(np.cos(ang)), np.mean(np.sin(ang)))
    # Rayleigh test (uniformity of direction) and axial version (2*ang for axis coherence)
    pR = np.exp(-len(ang)*R**2)
    Ra = np.hypot(np.mean(np.cos(2*ang)), np.mean(np.sin(2*ang)))
    pRa = np.exp(-len(ang)*Ra**2)
    ax_deg = (np.degrees(np.arctan2(np.mean(np.sin(2*ang)), np.mean(np.cos(2*ang))))/2) % 180
    print(f'{name:22s} N={len(ang):6d}  vector R={R:.3f} (Rayleigh p~{pR:.1e})   '
          f'axial R={Ra:.3f} (p~{pRa:.1e}, axis {ax_deg:.0f} deg)')
axs[1].set_xlabel('direction of mean residual vector [deg]'); axs[1].legend()
axs[1].set_title('direction coherence')

# (c) per-tile excess: is worsening concentrated in specific tiles?
tiles_u, tidx = np.unique(TILE, return_inverse=True)
src_tile = np.zeros(ncomp, dtype=int)
src_tile[gid] = tidx                      # any band's tile (sources live in one tile)
nt = len(tiles_u)
cnt_sub = np.bincount(src_tile[sub_i], minlength=nt)
cnt_all_src = np.bincount(src_tile[nb_src>=4], minlength=nt)
rate = SUB.sum()/ (nb_src>=4).sum()
exp = cnt_all_src*rate
m = exp>2
chi2_stat = np.sum((cnt_sub[m]-exp[m])**2/exp[m]); dof = m.sum()-1
p_tile = chi2.sf(chi2_stat, dof)
print(f'per-tile clustering: chi2/dof = {chi2_stat/dof:.2f} ({dof} tiles), p = {p_tile:.2e}')
axs[2].scatter(exp[m], cnt_sub[m], s=8, alpha=0.5, color='#444')
mx = max(exp[m].max(), cnt_sub[m].max())
axs[2].plot([0,mx],[0,mx],'k--',lw=1)
axs[2].set_xlabel('expected worseners per tile (uniform rate)')
axs[2].set_ylabel('observed'); axs[2].set_title(f'per-tile: chi2/dof={chi2_stat/dof:.2f}, p={p_tile:.1e}')
plt.tight_layout()
fig.savefig(OUTDIR/'nb31_spatial_direction.png',dpi=130,bbox_inches='tight')
print('saved', (OUTDIR/'nb31_spatial_direction.png').relative_to(REPO)); plt.show()

# within-source cross-band alignment: do a worsener's bands agree on the error direction?
al_sub, al_null = [], []
r2 = np.random.default_rng(3)
for i in sub_i:
    v = RESV[(gid==i)&W2]
    if len(v)>=2:
        u = v/np.linalg.norm(v,axis=1,keepdims=True)
        c = u@u.T; iu = np.triu_indices(len(u),1)
        al_sub.append(c[iu].mean())
if len(sub_i):
    # null: random pairs of W2 vectors from different sources
    vv = RESV[W2]; uu = vv/np.linalg.norm(vv,axis=1,keepdims=True)
    a = r2.integers(0,len(uu),20000); b = r2.integers(0,len(uu),20000)
    al_null = np.sum(uu[a]*uu[b],axis=1)[a!=b]
    fig2,ax2 = plt.subplots(figsize=(7,4.5))
    ax2.hist(al_sub, bins=40, density=True, histtype='step', lw=2, color='#c84b4b',
             label=f'within-source (med {np.median(al_sub):.2f})')
    ax2.hist(al_null, bins=40, density=True, histtype='step', lw=2, color='#888',
             label=f'across-source null (med {np.median(al_null):.2f})')
    ax2.set_xlabel('mean pairwise cos(angle) between band residual vectors')
    ax2.legend(); ax2.set_title('is the error direction shared across bands within a source?')
    print(f'within-source alignment median {np.median(al_sub):.3f} vs across-source null '
          f'{np.median(al_null):.3f}  (N={len(al_sub)} sources with >=2 W2 bands)')
    fig2.savefig(OUTDIR/'nb31_crossband_alignment.png',dpi=130,bbox_inches='tight')
    print('saved', (OUTDIR/'nb31_crossband_alignment.png').relative_to(REPO)); plt.show()

In [ ]:
# ---- does the head error point along the source -> companion axis? ----
# For each coherent worsener, unit vector to its nearest MER neighbor vs the mean head-residual
# vector. If companions pull (or deblending pushes) the head, |cos| piles up near 1.
# Caveat: the residual-vector frame may differ from (RA,Dec) by axis flips, which changes the
# SIGN of cos but not |cos| — the axial statistic is the robust one; the null comes from
# shuffling companion directions across sources.
mra = np.asarray(mc['ra'],float); mdec = np.asarray(mc['dec'],float)
dd2, ii2 = mt.query(np.column_stack([src_ra*cosd, src_dec]), k=2)
nbr = ii2[:,1]; nnd_all = dd2[:,1]*3600.
vto = np.column_stack([(mra[nbr]-src_ra)*cosd, mdec[nbr]-src_dec])
vto /= np.linalg.norm(vto, axis=1, keepdims=True)

def cos_with_err(idx, mv, rmax=2.0):
    m = idx[(nnd_all[idx]<rmax) & np.isfinite(mv[idx,0])]
    ue = mv[m]/np.linalg.norm(mv[m],axis=1,keepdims=True)
    return np.sum(ue*vto[m],axis=1), m
c_sub,_ = cos_with_err(sub_i, mean_vec)
c_ctl,_ = cos_with_err(ctl_i, mv1)
r4 = np.random.default_rng(4)
c_null = np.cos(r4.uniform(0,2*np.pi,20000))     # random relative angle

fig,axs = plt.subplots(1,2,figsize=(13,4.8))
for ax,f,lab in [(axs[0], lambda c:c, 'cos(angle)'), (axs[1], np.abs, '|cos(angle)|')]:
    for c,name,col in [(c_sub,f'worseners (N={len(c_sub)})','#c84b4b'),
                       (c_ctl,f'controls with W1 vec (N={len(c_ctl)})','#3b6fb6'),
                       (c_null,'random-direction null','#888')]:
        ax.hist(f(c), bins=30, density=True, histtype='step', lw=2, color=col, label=name)
    ax.set_xlabel(f'{lab} between head error and direction to nearest MER neighbor (<2")')
    ax.legend(fontsize=8)
plt.tight_layout()
print(f'worseners  median |cos|={np.median(np.abs(c_sub)):.2f}  frac |cos|>0.8: {(np.abs(c_sub)>0.8).mean():.1%}')
print(f'controls   median |cos|={np.median(np.abs(c_ctl)):.2f}  frac |cos|>0.8: {(np.abs(c_ctl)>0.8).mean():.1%}')
print(f'null       median |cos|={np.median(np.abs(c_null)):.2f}  frac |cos|>0.8: {(np.abs(c_null)>0.8).mean():.1%}')
print(f'signed: worseners frac cos>0 (toward companion) = {(c_sub>0).mean():.1%}')
# and the isolated worseners (no companion within 2"): how many, and are they bigger?
iso = sub_i[nnd_all[sub_i]>=2.0]
print(f'isolated worseners (NN>=2"): {len(iso)} of {len(sub_i)} '
      f'({len(iso)/len(sub_i):.0%}); median size {np.nanmedian(s_size[iso]):.2f}" '
      f'vs paired worseners {np.nanmedian(s_size[sub_i[nnd_all[sub_i]<2.0]]):.2f}"')
fig.savefig(OUTDIR/'nb31_companion_axis.png',dpi=130,bbox_inches='tight')
print('saved', (OUTDIR/'nb31_companion_axis.png').relative_to(REPO)); plt.show()

## Conclusions

**Headline (v11, all 9 bands, 790 tiles, 555k anchors):** raw 47.9 → head 13.3 mas median, 91.1% of
measurements improved.

**Are the worsened sources luck or a population? Both, and they separate cleanly.**

1. **The bulk is chance.** Most of the 8.9% worsened band-measurements are isolated single-band
   events: the k = 0, 1, 2 bins of the cross-band count match (or sit below) the chance null built
   from the empirical per-(band, S/N) rates. At high S/N the classical raw offset collapses toward
   the head's floor, so some worsening is unavoidable.

2. **A coherent failure population exists and is overwhelmingly non-chance.** 604 sources
   (1.6% of 37,928) worsen *materially* (res > raw + 10 mas and > 20 mas) in ≥ 3 distinct bands,
   vs 23 ± 5 expected by chance (×26; at k ≥ 4 the excess is ×400). The tile-overlap repeat test
   agrees (lift ×10–27).

3. **The failure is deterministic per source.** Within a source, the residual vector points the
   same way in every band (median pairwise cos = 0.997 vs 0.005 for the across-source null): the
   head consistently places each of these sources at one specific alternative position. Across
   sources the directions are isotropic (Rayleigh R = 0.03, p = 0.5), there is no tile clustering
   (chi²/dof = 1.12, p = 0.34) and no sky pattern — nothing instrumental or WCS-level survives in
   v11; the reason lives in the source's own pixels.

4. **What they are** (vs S/N-matched, same-band-count controls): galaxies (point-like fraction 1%
   for both), but *brighter* (VIS 23.1 vs 23.4) and *larger* (0.32" vs 0.28") at the same S/N,
   i.e. the extended, structured objects. They carry a ×1.7 excess of sub-arcsecond MER companions
   (7.6% vs 4.5% within 1"), and for the paired quarter (< 2" companion) the error lies along the
   source–companion axis (55% with |cos| > 0.8 vs 41% for random; 65% point *away* from the
   companion — a deblending-style push). The isolated 76% have no companion to blame: there the
   model's learned center simply disagrees with the classical light centroid, plausibly on
   asymmetric/clumpy morphology. (Axis-frame caveat: the residual frame may differ from RA/Dec by
   flips, which changes the sign but not |cos|.)

5. **The head partially knows.** Its predicted sigma is elevated on the failure population
   (median 18.3 vs 10.6 mas), so sigma-aware weighting already down-weights many of them, but the
   overlap is large — sigma alone does not isolate the population.

**Bottom line:** with v11 the worsened tail is no longer an instrument- or convention-level
systematic; what remains is a small (~1.6%), source-intrinsic population — extended galaxies,
part blends and part asymmetric morphology — where the model's notion of "center" differs
reproducibly from the classical centroid convention it is scored against.


## v10 comparison: why didn't this population show up before, and how much better is v11?

Identical pipeline on the v10 production archive
(`latent_position_q1_vissep/anchors_centernet_q1_vissep.npz`, same matched anchors) : dedup to one
measurement per (source, band), cross-band coherence vs the chance null, same subset definition
(≥ 4 bands, ≥ 3 material worsenings). Then a source-level cross-match of the two coherent
populations (fixed in v11 / persistent / new) and a composition comparison (S/N, VIS mag,
point-like fraction) to see what v10's population was made of that v11's is not.

In [ ]:
# ---- same pipeline on both archives ----
def pipeline(archpath, seed=1, nsims=200):
    dd = np.load(archpath, allow_pickle=True)
    rawv=[]; resv=[]; snr=[]; ra=[]; dec=[]; band=[]
    for bi,b in enumerate(BANDS):
        rr=np.asarray(dd[f'{b}_raw'])*1000.; hr=np.asarray(dd[f'{b}_head_resid'])*1000.
        rawv.append(rr); resv.append(hr); snr.append(np.asarray(dd[f'{b}_snr']))
        ra.append(np.asarray(dd[f'{b}_ra'])); dec.append(np.asarray(dd[f'{b}_dec']))
        band.append(np.full(len(rr), bi))
    rawv=np.vstack(rawv); resv=np.vstack(resv)
    snr=np.concatenate(snr); ra=np.concatenate(ra); dec=np.concatenate(dec); band=np.concatenate(band)
    rw=np.hypot(*rawv.T); rs=np.hypot(*resv.T)
    ok=(rw>0)&(rs>0)&np.isfinite(rw)&np.isfinite(rs)&np.isfinite(snr)&(snr>0)
    rawv,resv,rw,rs,snr,ra,dec,band=[a[ok] for a in (rawv,resv,rw,rs,snr,ra,dec,band)]
    med_raw, med_res, fimp = np.median(rw), np.median(rs), (rs<rw).mean()
    # group + dedup
    t=cKDTree(np.column_stack([ra*cosd,dec])); prs=t.query_pairs(0.3/3600.,output_type='ndarray')
    adj=coo_matrix((np.ones(len(prs)),(prs[:,0],prs[:,1])),shape=(len(ra),len(ra)))
    ncmp,g=connected_components(adj,directed=False)
    keysb=g.astype(np.int64)*16+band
    sel=np.lexsort((-snr,band,g)); kk=keysb[sel]
    fm=np.ones(len(sel),bool); fm[1:]=kk[1:]!=kk[:-1]; keep=sel[fm]
    rawv,resv,rw,rs,snr,ra,dec,band,g=[a[keep] for a in (rawv,resv,rw,rs,snr,ra,dec,band,g)]
    w1=rs>rw; w2=(rs>rw+10.)&(rs>20.); w3=rs>30.       # w3: raw-free (pure head failure)
    nbs=np.bincount(g,minlength=ncmp)
    k2=np.bincount(g,weights=w2,minlength=ncmp).astype(int)
    k3=np.bincount(g,weights=w3,minlength=ncmp).astype(int)
    sra=np.bincount(g,weights=ra,minlength=ncmp)/nbs; sdec=np.bincount(g,weights=dec,minlength=ncmp)/nbs
    ssnr=np.bincount(g,weights=snr,minlength=ncmp)/nbs
    # chance nulls for W2 and W3 (empirical per-(band, S/N-decile) rates)
    se=np.quantile(snr,np.linspace(0,1,11)); se[0]-=1; se[-1]+=1
    cl=band*10+np.clip(np.digitize(snr,se)-1,0,9)
    def rates(w):
        p=np.bincount(cl,weights=w)/np.maximum(np.bincount(cl),1); return p[cl]
    pi2, pi3 = rates(w2), rates(w3)
    r=np.random.default_rng(seed)
    null_k2=np.zeros((nsims,3)); null_k3=np.zeros((nsims,3))
    m4=nbs>=4
    for s in range(nsims):
        for pi_,nk in [(pi2,null_k2),(pi3,null_k3)]:
            w=r.random(len(pi_))<pi_
            ks=np.bincount(g,weights=w,minlength=ncmp).astype(int)
            for j,kt in enumerate((3,4,5)): nk[s,j]=(ks[m4]>=kt).sum()
    obs =[int((k2[m4]>=kt).sum()) for kt in (3,4,5)]
    obs3=[int((k3[m4]>=kt).sum()) for kt in (3,4,5)]
    sub =np.flatnonzero((nbs>=4)&(k2>=3))
    sub3=np.flatnonzero((nbs>=4)&(k3>=3))
    return dict(med_raw=med_raw,med_res=med_res,fimp=fimp,w1=w1,w2=w2,w3=w3,snr=snr,band=band,
                nbs=nbs,k2=k2,k3=k3,sra=sra,sdec=sdec,ssnr=ssnr,sub=sub,sub3=sub3,
                obs=obs,obs3=obs3,null_mean=null_k2.mean(0),null_std=null_k2.std(0),
                null3_mean=null_k3.mean(0),null3_std=null_k3.std(0),nsrc=ncmp,nmeas=len(rw))

ARCHS = {'v10 production': REPO/'models/checkpoints/latent_position_q1_vissep/anchors_centernet_q1_vissep.npz',
         'v11 plain':      ARCH}
R = {tag: pipeline(p_, seed=10+i) for i,(tag,p_) in enumerate(ARCHS.items())}
for tag,r_ in R.items():
    print(f'--- {tag} ---')
    print(f'  {r_["nmeas"]:,} deduped measurements, {r_["nsrc"]:,} sources | '
          f'raw {r_["med_raw"]:.1f} -> head {r_["med_res"]:.1f} mas, improved {r_["fimp"]:.1%} | '
          f'W1 {r_["w1"].mean():.1%}  W2 {r_["w2"].mean():.1%}')
    for j,kt in enumerate((3,4,5)):
        print(f'  W2 k>={kt}: observed {r_["obs"][j]:5d}  null {r_["null_mean"][j]:7.1f} +- {r_["null_std"][j]:5.1f}'
              f'   excess x{r_["obs"][j]/max(r_["null_mean"][j],1e-9):6.1f}')
    print(f'  coherent subset (k>=3): {len(r_["sub"]):,} sources')

In [ ]:
# ---- composition comparison + source-level cross-match of the two populations ----
v10, v11 = R['v10 production'], R['v11 plain']

def subset_props(r_):
    i = r_['sub']
    pl = match_generic(r_['sra'][i], r_['sdec'][i],
                       np.asarray(mc['ra'],float), np.asarray(mc['dec'],float),
                       (np.asarray(mc['point_like_flag'])==1).astype(float))
    vm = match_generic(r_['sra'][i], r_['sdec'][i],
                       np.asarray(mc['ra'],float), np.asarray(mc['dec'],float),
                       np.asarray(mc['mag_vis'],float))
    return r_['ssnr'][i], pl, vm
def match_generic(qra,qdec,cra,cdec,vals,rad=0.5):
    t=cKDTree(np.column_stack([cra*cosd,cdec]))
    dist,idx=t.query(np.column_stack([qra*cosd,qdec]),k=1)
    return np.where(dist*3600<rad, vals[idx], np.nan)

snr10,pl10,vm10 = subset_props(v10); snr11,pl11,vm11 = subset_props(v11)
print(f'coherent worseners: v10 {len(snr10):,}   v11 {len(snr11):,}   '
      f'(x{len(snr10)/len(snr11):.1f} reduction)')
print(f'point-like fraction: v10 {np.nanmean(pl10):.1%}   v11 {np.nanmean(pl11):.1%}')
print(f'median S/N: v10 {np.median(snr10):.1f}   v11 {np.median(snr11):.1f}')
print(f'fraction with S/N>50: v10 {(snr10>50).mean():.1%}   v11 {(snr11>50).mean():.1%}')
print(f'median VIS mag: v10 {np.nanmedian(vm10):.2f}   v11 {np.nanmedian(vm11):.2f}')

# cross-match subsets by position (same anchor set underneath)
t11=cKDTree(np.column_stack([v11['sra'][v11['sub']]*cosd, v11['sdec'][v11['sub']]]))
d10,_=t11.query(np.column_stack([v10['sra'][v10['sub']]*cosd, v10['sdec'][v10['sub']]]),k=1)
in_both = d10*3600<0.5
t10=cKDTree(np.column_stack([v10['sra'][v10['sub']]*cosd, v10['sdec'][v10['sub']]]))
d11,_=t10.query(np.column_stack([v11['sra'][v11['sub']]*cosd, v11['sdec'][v11['sub']]]),k=1)
new11 = d11*3600>=0.5
print(f'v10 worseners FIXED in v11: {(~in_both).sum():,} of {len(d10):,} ({(~in_both).mean():.0%})')
print(f'persistent in both: {in_both.sum():,}')
print(f'new in v11: {new11.sum():,} of {len(d11):,} ({new11.mean():.0%})')

fig,axs=plt.subplots(1,3,figsize=(17,5))
# (a) fraction W1-worsened vs S/N, measurement level
edges_s=np.geomspace(5,1000,14)
for (tag,r_),col in zip(R.items(),['#3b6fb6','#c84b4b']):
    sb=np.digitize(r_['snr'],edges_s)
    pts=[(np.median(r_['snr'][sb==i]),100*r_['w1'][sb==i].mean()) for i in range(1,len(edges_s)) if (sb==i).sum()>500]
    axs[0].plot([x[0] for x in pts],[x[1] for x in pts],'o-',color=col,label=tag)
    pts2=[(np.median(r_['snr'][sb==i]),100*r_['w2'][sb==i].mean()) for i in range(1,len(edges_s)) if (sb==i).sum()>500]
    axs[0].plot([x[0] for x in pts2],[x[1] for x in pts2],'s--',color=col,alpha=0.6,mfc='none')
axs[0].set(xscale='log',xlabel='S/N',ylabel='fraction worsened [%]')
axs[0].set_title('worsening vs S/N (solid W1, dashed W2)')
axs[0].axvline(100,color='k',ls=':',lw=1); axs[0].legend()
# (b) S/N distribution of the coherent populations
bins_s=np.geomspace(3,3000,35)
axs[1].hist(snr10,bins=bins_s,histtype='step',lw=2,color='#3b6fb6',label=f'v10 (N={len(snr10):,})')
axs[1].hist(snr11,bins=bins_s,histtype='step',lw=2,color='#c84b4b',label=f'v11 (N={len(snr11):,})')
axs[1].set(xscale='log',xlabel='mean S/N of coherent worseners',ylabel='N sources')
axs[1].set_title('who the coherent worseners are'); axs[1].legend()
# (c) VIS mag distribution
bins_m=np.linspace(16,26,40)
axs[2].hist(vm10[np.isfinite(vm10)],bins=bins_m,histtype='step',lw=2,color='#3b6fb6',label='v10')
axs[2].hist(vm11[np.isfinite(vm11)],bins=bins_m,histtype='step',lw=2,color='#c84b4b',label='v11')
axs[2].set(xlabel='VIS mag of coherent worseners',ylabel='N sources'); axs[2].legend()
plt.tight_layout()
fig.savefig(OUTDIR/'nb31_v10_vs_v11.png',dpi=130,bbox_inches='tight')
print('saved',(OUTDIR/'nb31_v10_vs_v11.png').relative_to(REPO)); plt.show()

### v10 vs v11: takeaways

**The galaxy population was already there in v10 — it just wasn't what anyone was looking at.**
Running the identical coherence test on v10 finds essentially the same population: 622 coherent
worseners vs 604 in v11, with 486 (80%) the *same sources* in both, same median VIS mag (23.0 vs
23.1), same median S/N (14.5 vs 13.5). This population is not a v10 defect and not something v11
introduced: it is a stable property of how a learned center disagrees with the classical centroid
on extended galaxies. It went unnoticed because per-source it hides at S/N ≈ 13, where a single
worsened band looks exactly like chance; it only becomes visible when you demand the worsening
repeat across ≥ 3 bands. Earlier worsener work selected on the *bright* tail, which was dominated
by a different problem.

**What v11 actually fixed is the bright end.** At S/N ≈ 100 the worsened fraction drops from 39%
to 25% (W1) and the material rate from 7% to 3%; the high-S/N members of the coherent population
fall from 4.8% to 1.8% (S/N > 50), and its point-like fraction from 2.0% to 1.2%. In other words,
v11 removed the star/bright component and left the faint extended-galaxy component untouched.

**Net v10 → v11:** median head residual 13.4 → 13.3 mas, improved fraction 90.8% → 91.1%,
material-worsening rate 3.3% → 3.2%, coherent population 622 → 604 (−3%); of v10's population 22%
are fixed, but a comparable 20% are new in v11 — consistent with a persistent core (80%) plus a
noise-level exchange at the k ≥ 3 selection boundary.

**Implication:** further gains on the worsened tail won't come from foundation-level fixes; they
require either a head that understands morphology-dependent center conventions (or is trained
against a convention-consistent target for extended sources), or masking/down-weighting via the
head's own sigma, which is already elevated on this population.

## Snapshots: where the head and the classical centroid put these galaxies

Position algebra (from `eval_latent_position.py`, tangent-plane arcsec, components (Δα·cosδ, Δδ)):
- archive `ra/dec` = **classical band centroid** (projected to the VIS frame),
- `raw` = VIS label − band centroid, so **VIS/MER label = anchor + raw**,
- `head_resid` = label − head estimate, so **head position = label − head_resid**.

Each stamp: VIS image (0.1"/px), 4" wide, with a 1" zoom inset. Markers: white **+** = VIS
label (the convention we score against), blue dots = classical per-band centroids, red dots =
head per-band positions, orange circles = MER catalog neighbors. Two galleries: worseners with
a companion < 2", and isolated ones.

In [ ]:
# ---- absolute positions per measurement + gallery selection ----
lab_ra  = RA  + (RAWV[:,0]/3.6e6)/cosd      # RAWV, RESV in mas
lab_dec = DEC +  RAWV[:,1]/3.6e6
hd_ra   = lab_ra  - (RESV[:,0]/3.6e6)/cosd
hd_dec  = lab_dec -  RESV[:,1]/3.6e6

# sanity: the VIS label should be (nearly) the same in every band of a source
mlra = np.bincount(gid, weights=lab_ra, minlength=ncomp)/nb_src
mldec= np.bincount(gid, weights=lab_dec,minlength=ncomp)/nb_src
dev = np.hypot((lab_ra-mlra[gid])*cosd,(lab_dec-mldec[gid]))*3.6e6
print(f'label-position per-source scatter: median {np.median(dev[nb_src[gid]>=4]):.1f} mas (sanity, ~0 expected)')

# group-span diagnostic: the 0.3" linking can chain a blended pair, in which case different
# bands may have matched DIFFERENT components (matching ambiguity, not a head error).
span = np.zeros(ncomp)
for i in sub_i:
    m = gid==i
    dx=(lab_ra[m]-mlra[i])*cosd; dy=lab_dec[m]-mldec[i]
    span[i] = 2*np.hypot(dx,dy).max()*3600.
amb = span[sub_i] > 0.5
print(f'label-position span of worsener groups: median {np.median(span[sub_i]):.2f}", '
      f'>0.5" (cross-band matching ambiguity candidates): {amb.sum()} of {len(sub_i)} ({amb.mean():.0%})')

# per-source W2 residual magnitude + tiles of the group (S/N-sorted, for VIS stamp fallback)
w2res = np.zeros(ncomp); best_tile = np.empty(ncomp, dtype=object); tile_opts = {}
for i in sub_i:
    m = gid==i
    w2res[i] = np.median(RES[m & W2]) if (m & W2).any() else 0
    o = np.argsort(-SNR[m]); tile_opts[i] = list(dict.fromkeys(TILE[m][o]))
    best_tile[i] = tile_opts[i][0]
order = sub_i[np.argsort(-(k2[sub_i]*1e4 + w2res[sub_i]))]     # high-k first, then big residuals
clean = span[order] <= 0.3                                     # unambiguous groups for the gallery
paired   = [i for i,c in zip(order,clean) if c and nnd_all[i] <  2.0][:8]
isolated = [i for i,c in zip(order,clean) if c and nnd_all[i] >= 2.0][:12]
print(f'gallery: {len(paired)} paired + {len(isolated)} isolated '
      f'(k of shown sources: {sorted(set(k2[paired+isolated].tolist()))})')

In [ ]:
# ---- cutout galleries ----
import sys
if str(REPO/'models') not in sys.path: sys.path.insert(0, str(REPO/'models'))
from astrometry2.source_matching import safe_header_from_card_string
from astropy.wcs import WCS as AWCS

_tile_cache = {}
def get_vis(tile):
    if tile not in _tile_cache:
        e = np.load(REPO/f'data/euclid_tiles_all/{tile}_euclid.npz', allow_pickle=True)
        _tile_cache[tile] = (np.asarray(e['img_VIS'], float),
                             AWCS(safe_header_from_card_string(e['wcs_VIS'].item())))
    return _tile_cache[tile]

mra_all = np.asarray(mc['ra'],float); mdec_all = np.asarray(mc['dec'],float)
def pick_tile(i, half_as=2.0):
    # first tile (S/N-sorted) whose VIS stamp is mostly finite
    h = int(round(half_as/0.1))
    for t in tile_opts.get(i, [best_tile[i]]):
        img, w = get_vis(t)
        x0, y0 = w.wcs_world2pix(mlra[i], mldec[i], 0)
        xi, yi = int(round(float(x0))), int(round(float(y0)))
        if not (h<=xi<img.shape[1]-h and h<=yi<img.shape[0]-h): continue
        st = img[yi-h:yi+h, xi-h:xi+h]
        if np.isfinite(st).mean() > 0.7: return t
    return None

def draw_source(ax, i, half_as=2.0, inset_as=0.5):
    t = pick_tile(i, half_as)
    if t is None: return False
    img, w = get_vis(t)
    x0, y0 = w.wcs_world2pix(mlra[i], mldec[i], 0)
    h = int(round(half_as/0.1))
    xi, yi = int(round(float(x0))), int(round(float(y0)))
    st = img[yi-h:yi+h, xi-h:xi+h]
    lo = np.nanmedian(st); sc = np.nanpercentile(st-lo, 99.5)
    st = np.where(np.isfinite(st), st, lo)
    ax.imshow(np.arcsinh((st-lo)/max(sc/10,1e-9)), origin='lower', cmap='gray',
              extent=[xi-h-0.5, xi+h-0.5, yi-h-0.5, yi+h-0.5])
    m = gid==i
    cx, cy = w.wcs_world2pix(RA[m], DEC[m], 0)          # classical band centroids
    hx, hy = w.wcs_world2pix(hd_ra[m], hd_dec[m], 0)    # head positions
    near = np.hypot((mra_all-mlra[i])*cosd, mdec_all-mldec[i])*3600
    nb = (near>0.3)&(near<half_as*1.4)
    nx, ny = w.wcs_world2pix(mra_all[nb], mdec_all[nb], 0)
    for a in [ax]:
        a.scatter(nx, ny, s=260, facecolors='none', edgecolors='orange', lw=1.4)
        a.scatter(cx, cy, s=14, c='#4da6ff', lw=0, alpha=0.9)
        a.scatter(hx, hy, s=14, c='#ff5544', lw=0, alpha=0.9)
        a.plot(x0, y0, '+', c='white', ms=11, mew=1.6)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f'k={k2[i]}, S/N={src_snr[i]:.0f}, NN={nnd_all[i]:.1f}", '
                 f'$\\langle$head off$\\rangle$={np.mean(RES[m]):.0f} mas', fontsize=9)
    # zoom inset
    axz = ax.inset_axes([0.62, 0.62, 0.36, 0.36])
    hz = inset_as/0.1
    axz.imshow(np.arcsinh((st-lo)/max(sc/10,1e-9)), origin='lower', cmap='gray',
               extent=[xi-h-0.5, xi+h-0.5, yi-h-0.5, yi+h-0.5], interpolation='nearest')
    axz.scatter(cx, cy, s=26, c='#4da6ff', lw=0)
    axz.scatter(hx, hy, s=26, c='#ff5544', lw=0)
    axz.plot(x0, y0, '+', c='white', ms=12, mew=1.8)
    axz.set_xlim(x0-hz, x0+hz); axz.set_ylim(y0-hz, y0+hz)
    axz.set_xticks([]); axz.set_yticks([])
    for sp in axz.spines.values(): sp.set_color('yellow')
    return True

for name, group, (nr, nc_) in [('paired', paired, (2,4)), ('isolated', isolated, (3,4))]:
    fig, axs = plt.subplots(nr, nc_, figsize=(4*nc_, 4.15*nr))
    for ax, i in zip(axs.ravel(), group):
        if not draw_source(ax, i): ax.axis('off')
    for ax in axs.ravel()[len(group):]: ax.axis('off')
    fig.suptitle(f'coherent worseners, {name}  —  white + = VIS/MER label, '
                 f'blue = classical band centroids, red = head, orange = MER neighbors '
                 f'(inset: 1" zoom)', fontsize=12, y=1.0)
    plt.tight_layout()
    fig.savefig(OUTDIR/f'nb31_gallery_{name}.png', dpi=120, bbox_inches='tight')
    print('saved', (OUTDIR/f'nb31_gallery_{name}.png').relative_to(REPO))
    plt.show()

## Who is right, the head or the classical centroid?

There is no external truth for a galaxy's center (Gaia is stars-only), so two operational tests:

1. **Cross-band repeatability** (archive-only, all sources): a good position estimator, applied to
   9 independent images of the same object, should agree with itself. Per source we compare the
   RMS scatter of the classical band centroids about their mean vs the same for the head's
   positions. Whichever clusters tighter is measuring a better-defined point on these objects.
2. **What does each position sit on?** (pixel-level, sample of worseners + same-tile controls):
   distance of the head position, the VIS label, and the classical centroids to (a) the VIS
   brightest-pixel peak (sub-pixel refined) and (b) the VIS flux-weighted centroid in a 0.75"
   aperture. This says what each estimator is latching onto.

In [ ]:
# ---- test 1: cross-band repeatability of absolute positions ----
def per_src_rms(pra, pdec):
    mr = np.bincount(gid, weights=pra, minlength=ncomp)/nb_src
    md_ = np.bincount(gid, weights=pdec, minlength=ncomp)/nb_src
    d2 = ((pra-mr[gid])*cosd)**2 + (pdec-md_[gid])**2
    return np.sqrt(np.bincount(gid, weights=d2, minlength=ncomp)/nb_src)*3.6e6  # mas

rms_cl = per_src_rms(RA, DEC)          # classical band centroids
rms_hd = per_src_rms(hd_ra, hd_dec)    # head positions
fig,axs = plt.subplots(1,2,figsize=(13,5))
bins = np.geomspace(1,300,40)
for ax, ii, tag in [(axs[0], sub_i, 'coherent worseners'), (axs[1], ctl_i, 'S/N-matched controls')]:
    ax.hist(rms_cl[ii], bins=bins, histtype='step', lw=2, color='#4da6ff',
            label=f'classical (med {np.median(rms_cl[ii]):.0f} mas)')
    ax.hist(rms_hd[ii], bins=bins, histtype='step', lw=2, color='#ff5544',
            label=f'head (med {np.median(rms_hd[ii]):.0f} mas)')
    ax.set(xscale='log', xlabel='cross-band RMS about own mean [mas]', title=tag); ax.legend()
plt.tight_layout()
iso_i = sub_i[nnd_all[sub_i]>=2.0]; pr_i = sub_i[nnd_all[sub_i]<2.0]
for ii,tag in [(sub_i,'worseners (all)   '),(iso_i,'worseners isolated'),
               (pr_i,'worseners paired  '),(ctl_i,'controls          ')]:
    print(f'{tag}: classical RMS {np.median(rms_cl[ii]):5.1f} vs head RMS {np.median(rms_hd[ii]):5.1f} mas'
          f'   (head tighter: {(rms_hd[ii]<rms_cl[ii]).mean():.0%}, N={len(ii)})')
fig.savefig(OUTDIR/'nb31_repeatability.png', dpi=130, bbox_inches='tight')
print('saved',(OUTDIR/'nb31_repeatability.png').relative_to(REPO)); plt.show()

In [ ]:
# ---- test 2: what does each position sit on in the VIS image? ----
# sample worseners tile by tile (cap tiles for runtime) + same-tile controls
def refine_peak(st):
    yy,xx = np.unravel_index(np.argmax(st), st.shape)
    if 0<yy<st.shape[0]-1 and 0<xx<st.shape[1]-1:
        dx = 0.5*(st[yy,xx-1]-st[yy,xx+1])/(st[yy,xx-1]-2*st[yy,xx]+st[yy,xx+1]+1e-12)
        dy = 0.5*(st[yy-1,xx]-st[yy+1,xx])/(st[yy-1,xx]-2*st[yy,xx]+st[yy+1,xx]+1e-12)
        return xx+np.clip(dx,-1,1), yy+np.clip(dy,-1,1)
    return float(xx), float(yy)

def pixel_metrics(i):
    t = pick_tile(i, half_as=1.0)
    if t is None: return None
    img, w = get_vis(t)
    x0,y0 = w.wcs_world2pix(mlra[i], mldec[i], 0)
    xi,yi = int(round(float(x0))), int(round(float(y0)))
    h=10                                     # 1" search box for the peak
    if not (h<=xi<img.shape[1]-h and h<=yi<img.shape[0]-h): return None
    st = img[yi-h:yi+h+1, xi-h:xi+h+1]
    if np.isfinite(st).mean() < 0.9: return None
    bg = np.nanmedian(img[max(0,yi-30):yi+30, max(0,xi-30):xi+30])
    st = np.where(np.isfinite(st), st, bg) - bg
    inner = st[h-6:h+7, h-6:h+7]             # peak within 0.6"
    px,py = refine_peak(inner); px+=xi-6; py+=yi-6
    # flux-weighted centroid, r<=0.75"
    Y,X = np.mgrid[yi-h:yi+h+1, xi-h:xi+h+1]
    ap = np.hypot(X-x0, Y-y0) <= 7.5
    wgt = np.clip(st,0,None)*ap
    if wgt.sum()<=0: return None
    cx = (X*wgt).sum()/wgt.sum(); cy=(Y*wgt).sum()/wgt.sum()
    # head mean position + classical mean, in this tile's pixels
    m = gid==i
    hx,hy = w.wcs_world2pix(np.mean(hd_ra[m]), np.mean(hd_dec[m]), 0)
    ax_,ay_ = w.wcs_world2pix(np.mean(RA[m]), np.mean(DEC[m]), 0)
    def d(xa,ya,xb,yb): return float(np.hypot(xa-xb,ya-yb)*100)   # mas
    return dict(head_peak=d(hx,hy,px,py), lab_peak=d(x0,y0,px,py), cl_peak=d(ax_,ay_,px,py),
                head_cen=d(hx,hy,cx,cy),  lab_cen=d(x0,y0,cx,cy),  cl_cen=d(ax_,ay_,cx,cy))

r5 = np.random.default_rng(5)
tiles_sub = {}
for i in sub_i: tiles_sub.setdefault(best_tile[i], []).append(i)
use_tiles = list(tiles_sub)[:150]
samp_w = [i for t in use_tiles for i in tiles_sub[t]]
# same-tile controls: need best_tile for controls too
ctl_set = set(ctl_i.tolist()); samp_c=[]
tile_of_ctl = {}
for i in ctl_i:
    m = gid==i
    tile_of_ctl[i] = TILE[m][np.argmax(SNR[m])]
for i,t in tile_of_ctl.items():
    if t in use_tiles: samp_c.append(i)
samp_c = r5.permutation(samp_c)[:len(samp_w)].tolist()
for i in samp_c: best_tile[i] = tile_of_ctl[i]
res_w = [(i,m_) for i,m_ in ((i, pixel_metrics(i)) for i in samp_w) if m_]
res_c = [m_ for m_ in (pixel_metrics(i) for i in samp_c) if m_]
res_iso = [m_ for i,m_ in res_w if nnd_all[i]>=2.0]
res_pr  = [m_ for i,m_ in res_w if nnd_all[i]<2.0]
print(f'pixel test: {len(res_iso)} isolated + {len(res_pr)} paired worseners, '
      f'{len(res_c)} controls ({len(use_tiles)} tiles)')
def med(rs,k): return np.median([r_[k] for r_ in rs])
print(f'{"":34s}{"head":>7s}{"VIS label":>11s}{"classical":>11s}')
for k1_,k2_,k3_,lab in [('head_peak','lab_peak','cl_peak','dist to VIS peak [mas]'),
                        ('head_cen','lab_cen','cl_cen','dist to VIS centroid   ')]:
    for rs,tag in [(res_iso,'worseners ISOLATED'),(res_pr,'worseners paired  '),(res_c,'controls          ')]:
        print(f'{tag} {lab}: {med(rs,k1_):6.0f} {med(rs,k2_):10.0f} {med(rs,k3_):10.0f}')

fig,axs=plt.subplots(1,2,figsize=(13,5))
bins=np.geomspace(2,1000,40)
for ax,(ka,kb,ttl) in zip(axs,[('head_peak','lab_peak','distance to VIS peak'),
                               ('head_cen','lab_cen','distance to VIS flux centroid (0.75")')]):
    ax.hist([r_[ka] for r_ in res_iso],bins=bins,histtype='step',lw=2,color='#ff5544',label='head (isolated worseners)')
    ax.hist([r_[kb] for r_ in res_iso],bins=bins,histtype='step',lw=2,color='#999999',label='VIS label (isolated worseners)')
    ax.hist([r_[ka] for r_ in res_pr],bins=bins,histtype='step',lw=2,color='#a030d0',label='head (paired worseners)')
    ax.hist([r_[ka] for r_ in res_c],bins=bins,histtype='step',lw=2,color='#ff5544',ls=':',label='head (controls)')
    ax.set(xscale='log',xlabel='distance [mas]',title=ttl); ax.legend(fontsize=9)
plt.tight_layout()
fig.savefig(OUTDIR/'nb31_peak_centroid_test.png',dpi=130,bbox_inches='tight')
print('saved',(OUTDIR/'nb31_peak_centroid_test.png').relative_to(REPO)); plt.show()

## Is it still chance? The correlated-error loophole, closed

A fair objection to the coherence test: the head reads ONE shared fused representation per tile,
so its errors are correlated across bands *by construction*. Under the independence null, one
unlucky per-source draw of the head could then masquerade as k = 8 "coherent" worsenings. The
×26 excess alone cannot exclude that.

The test that can: **v10 and v11 are independently trained models** (different foundation,
different head weights). A random per-source head draw is redrawn between trainings, so under the
luck hypothesis the two coherent populations should overlap only at the chance level. Shared
ingredients (the classical raw offsets and the labels are identical in both runs) cannot create
excess overlap on their own, because the material-worsening definition requires the *head* to be
> 20 mas off — and that draw is independent across trainings unless it is pinned by the pixels.

Below: (a) per-band vectors for example gallery sources — the classical raw offsets (blue,
independent measurements) scatter, the head residuals (red) cluster; (b) the cross-training
overlap vs its hypergeometric expectation; (c) which gallery sources were already worseners in
v10.

In [ ]:
# ---- (a) per-band vectors for 3 isolated gallery sources ----
fig,axs = plt.subplots(1,3,figsize=(15,5.2))
for ax,i in zip(axs, isolated[:3]):
    m = gid==i
    ax.scatter(RAWV[m,0], RAWV[m,1], s=60, c='#4da6ff', label='raw = classical$-$label (per band)')
    ax.scatter(RESV[m,0], RESV[m,1], s=60, c='#ff5544', marker='s', label='head resid (per band)')
    for r_ in (20,40,80): ax.add_patch(plt.Circle((0,0), r_, fill=False, ec='0.75', lw=0.8))
    ax.axhline(0,color='0.85',lw=0.8); ax.axvline(0,color='0.85',lw=0.8)
    ax.plot(0,0,'+',c='k',ms=12,mew=1.5)
    lim = max(100, 1.2*np.abs(np.r_[RAWV[m],RESV[m]]).max())
    ax.set_xlim(-lim,lim); ax.set_ylim(-lim,lim); ax.set_aspect('equal')
    ax.set_xlabel('$\\Delta\\alpha\\cos\\delta$ [mas]'); ax.set_ylabel('$\\Delta\\delta$ [mas]')
    ax.set_title(f'k={k2[i]}, S/N={src_snr[i]:.0f}, $\\langle$head off$\\rangle$={np.mean(RES[m]):.0f} mas',
                 fontsize=10)
axs[0].legend(fontsize=9, loc='lower left')
fig.suptitle('independent classical measurements scatter around the label (+); the head repeats one offset',
             fontsize=12)
plt.tight_layout()
fig.savefig(OUTDIR/'nb31_perband_vectors.png',dpi=130,bbox_inches='tight')
print('saved',(OUTDIR/'nb31_perband_vectors.png').relative_to(REPO)); plt.show()

# ---- (b) cross-training persistence vs chance ----
from scipy.stats import hypergeom
v10s, v11s = R['v10 production'], R['v11 plain']
p10 = np.column_stack([v10s['sra'][v10s['sub']]*cosd, v10s['sdec'][v10s['sub']]])
p11 = np.column_stack([v11s['sra'][v11s['sub']]*cosd, v11s['sdec'][v11s['sub']]])
d,_ = cKDTree(p11).query(p10, k=1)
n_overlap = int((d*3600<0.5).sum())
M = int((v11s['nbs']>=4).sum())                    # eligible pool (>=4 bands)
exp = len(p10)*len(p11)/M
p_hyp = hypergeom.sf(n_overlap-1, M, len(p10), len(p11))
print(f'cross-training overlap: {n_overlap} of {len(p10)}/{len(p11)} '
      f'(eligible pool {M:,}) — chance expects {exp:.0f}; enrichment x{n_overlap/exp:.0f}, '
      f'hypergeometric p ~ {p_hyp:.1e}')
kk_hi = int(((v11s['nbs']>=4)&(v11s['k2']>=5)).sum())
print(f'(and the k>=5 core alone: {kk_hi} sources vs {v11s["null_mean"][2]:.1f} expected by chance)')

# ---- (c) were the gallery sources already worseners in v10? ----
t10 = cKDTree(p10)
for name, group in [('paired',paired),('isolated',isolated)]:
    q = np.column_stack([mlra[group]*cosd, mldec[group]])
    dg,_ = t10.query(q, k=1)
    flags = ['v10-too' if x*3600<0.5 else 'v11-only' for x in dg]
    print(f'{name} gallery: ' + ', '.join(f'{k2[i]}k/{f}' for i,f in zip(group,flags)))

### The frozen-raw objection: what if the classical luck is shared between runs?

Sharp objection: the classical raw offsets and the labels are **identical** in v10 and v11 (same
images, same classical pipeline). If "worsened" status came from raws that happen to be tight,
that luck is frozen into both runs, and cross-training persistence would say nothing about the
model. Two answers:

1. **Are the worseners raw-lucky?** Compare the raw-offset distribution of the coherent worseners'
   band measurements to the S/N-matched controls'. If they were selected for tight classical
   positions, their raws would be systematically smaller.
2. **Remove raw from the definition entirely.** W3 = head residual > 30 mas, absolute — no
   reference to raw anywhere. Redo the coherence null and the cross-training persistence with W3.
   Classical luck cannot enter this selection at any point; whatever survives is a property of the
   head + pixels.

In [ ]:
# ---- (1) are the worseners selected for tight raws? ----
msub = np.isin(gid, sub_i); mctl = np.isin(gid, ctl_i)
print(f'band-level medians   raw      head-res')
print(f'  worseners        {np.median(RAW[msub]):5.1f} mas {np.median(RES[msub]):8.1f} mas')
print(f'  matched controls {np.median(RAW[mctl]):5.1f} mas {np.median(RES[mctl]):8.1f} mas')

# ---- (2) raw-free definition W3 = res > 30 mas: coherence + persistence ----
for tag,r_ in R.items():
    print(f'--- {tag} (W3: head res > 30 mas, raw-free) ---')
    for j,kt in enumerate((3,4,5)):
        print(f'  k>={kt}: observed {r_["obs3"][j]:5d}  null {r_["null3_mean"][j]:7.1f} '
              f'+- {r_["null3_std"][j]:5.1f}   excess x{r_["obs3"][j]/max(r_["null3_mean"][j],1e-9):6.1f}')
q10 = np.column_stack([v10s['sra'][v10s['sub3']]*cosd, v10s['sdec'][v10s['sub3']]])
q11 = np.column_stack([v11s['sra'][v11s['sub3']]*cosd, v11s['sdec'][v11s['sub3']]])
d3,_ = cKDTree(q11).query(q10, k=1)
n_ov3 = int((d3*3600<0.5).sum())
M3 = int((v11s['nbs']>=4).sum())
exp3 = len(q10)*len(q11)/M3
p3 = hypergeom.sf(n_ov3-1, M3, len(q10), len(q11))
print(f'W3 cross-training overlap: {n_ov3} of {len(q10)}/{len(q11)} — chance expects {exp3:.0f}; '
      f'enrichment x{n_ov3/max(exp3,1e-9):.0f}, hypergeometric p ~ {p3:.1e}')
# and the W3 population vs the W2 one (same sources?)
dq,_ = cKDTree(p11).query(q11, k=1)
print(f'W3(v11) sources also in the W2(v11) population: {(dq*3600<0.5).mean():.0%}')

### Resolution of the frozen-raw objection

1. **The worseners are not raw-lucky.** Their band-level raw offsets are only mildly smaller than
   matched controls' (median 30.6 vs 37.7 mas, −19%), while their head residuals are ×4.7 larger
   (43.8 vs 9.4 mas). The selection is overwhelmingly driven by the head being off, not by the
   classical being tight. (The mild raw deficit is real and concedes a grain of the objection: a
   slightly-better-than-typical classical makes the res > raw threshold marginally easier — but it
   is a second-order effect.)

2. **The raw-free selection (W3: head res > 30 mas, raw never consulted) is still coherent and
   still persistent across independent trainings.** W3 selects a larger, mostly-faint population
   (~5,650 sources; at that S/N a 30 mas head residual is common), so the k ≥ 3 enrichment is
   modest (×1.1), but at k ≥ 4 it is ×2.7 and at k ≥ 5 ×8 — far outside the simulation scatter.
   Cross-training: 93% of v10's W3-coherent sources are also v11's (5,293 of 5,687), vs 29%
   expected by chance (×3, hypergeometric p ≈ 0). Classical luck cannot enter W3 at any point, and
   raw is frozen anyway — the only re-randomized ingredient between v10 and v11 is the head, so
   this overlap can only come from the head's errors being pinned to the source's pixels.

3. W3 and W2 are different populations (only 10% of W3 sources are W2 members): W3 = "the head is
   chronically imprecise here" (mostly faint, classical struggles too); W2 = "the head is
   materially worse *than the classical*" — the failure population of this notebook. Both are
   source-pinned; only W2 indicts the head relative to the classical convention.

## Mechanism hunt: chromatic centers and a scatter-refined population

Two hypotheses to test rather than assume:

1. **Color-dependent centers.** A galaxy's center can genuinely move with wavelength (bulge vs
   disk, blue clumps, dust). If so, the classical band centroids should drift *systematically*
   with wavelength, not scatter randomly — measurable as a per-source chromatic slope
   (mas/μm, u → H spans 0.37–1.77 μm). Questions: are the worseners more chromatic than matched
   controls? and does the head's offset lie along the chromatic axis (e.g. the head reporting a
   redder / flux-weighted cross-band center while the label is the VIS-band convention)?
2. **Scatter-refined membership.** A source whose head positions scatter widely across bands is
   not a *coherent* worsener even if it passes k ≥ 3 — its failures look stochastic. Split the
   population by the head's cross-band RMS (tight < 15 mas vs loose) and check which part carries
   the cross-training persistence — the loose part is the one to discount.

In [ ]:
# ---- chromatic slope per source + scatter split ----
LAM = np.array([0.368,0.480,0.622,0.754,0.869,0.971,1.081,1.273,1.773])  # um, BANDS order

def chroma(i):
    m = gid==i
    if m.sum() < 5: return None
    x = LAM[BAND[m]]
    if x.max()-x.min() < 0.5: return None
    xc = x - x.mean()
    dxy = -RAWV[m]                                  # classical - label, mas
    sx = np.polyfit(xc, dxy[:,0], 1); sy = np.polyfit(xc, dxy[:,1], 1)
    resid = dxy - np.column_stack([np.polyval(sx,xc), np.polyval(sy,xc)])
    rms = np.sqrt((resid**2).sum(1).mean())
    slope = np.array([sx[0], sy[0]])                # mas/um
    sig = np.linalg.norm(slope)*(x.max()-x.min())/(rms+1e-9)
    hvec = -RESV[m].mean(0)                         # head - label, mas
    cosal = np.dot(hvec, slope)/(np.linalg.norm(hvec)*np.linalg.norm(slope)+1e-12)
    # head vs blue/red classical centers
    nis = BAND[m]>=6; rub = BAND[m]<6
    d_nisp = d_rub = np.nan
    if nis.any() and rub.any():
        d_nisp = np.linalg.norm(hvec - (-RAWV[m][nis]).mean(0))
        d_rub  = np.linalg.norm(hvec - (-RAWV[m][rub]).mean(0))
    return dict(slope=np.linalg.norm(slope), sig=sig, cos=cosal,
                d_lab=np.linalg.norm(hvec), d_nisp=d_nisp, d_rub=d_rub)

CH = {}
for i in np.concatenate([sub_i, ctl_i]):
    c_ = chroma(i)
    if c_: CH[i] = c_
def arr(ii,k): return np.array([CH[i][k] for i in ii if i in CH])

fig,axs = plt.subplots(1,3,figsize=(16,4.8))
axs[0].hist(arr(sub_i,'slope'), bins=np.geomspace(1,300,35), histtype='step', lw=2,
            color='#c84b4b', density=True, label=f'worseners (med {np.median(arr(sub_i,"slope")):.0f})')
axs[0].hist(arr(ctl_i,'slope'), bins=np.geomspace(1,300,35), histtype='step', lw=2,
            color='#3b6fb6', density=True, label=f'controls (med {np.median(arr(ctl_i,"slope")):.0f})')
axs[0].set(xscale='log', xlabel='|chromatic slope| [mas/$\\mu$m]'); axs[0].legend(fontsize=9)
axs[1].hist(arr(sub_i,'sig'), bins=np.linspace(0,8,33), histtype='step', lw=2, color='#c84b4b',
            density=True, label=f'worseners (frac>2: {(arr(sub_i,"sig")>2).mean():.0%})')
axs[1].hist(arr(ctl_i,'sig'), bins=np.linspace(0,8,33), histtype='step', lw=2, color='#3b6fb6',
            density=True, label=f'controls (frac>2: {(arr(ctl_i,"sig")>2).mean():.0%})')
axs[1].set(xlabel='chromatic significance (slope$\\times$span / band scatter)'); axs[1].legend(fontsize=9)
sig_w = arr(sub_i,'sig'); cos_w = arr(sub_i,'cos')
axs[2].hist(cos_w[sig_w>2], bins=np.linspace(-1,1,25), histtype='step', lw=2, color='#c84b4b',
            density=True, label='worseners, sig>2')
axs[2].hist(cos_w[sig_w<=1], bins=np.linspace(-1,1,25), histtype='step', lw=2, color='#888',
            density=True, label='worseners, sig<=1')
axs[2].set(xlabel='cos(head offset, chromatic axis)'); axs[2].legend(fontsize=9)
plt.tight_layout()
fig.savefig(OUTDIR/'nb31_chromatic.png', dpi=130, bbox_inches='tight')
print('saved',(OUTDIR/'nb31_chromatic.png').relative_to(REPO))
print(f'chromatic worseners (sig>2): head aligned with axis |cos|>0.7: '
      f'{(np.abs(cos_w[sig_w>2])>0.7).mean():.0%} (random: 49%); '
      f'toward RED end (cos>0.7): {(cos_w[sig_w>2]>0.7).mean():.0%}')
dn, dr, dl = arr(sub_i,'d_nisp'), arr(sub_i,'d_rub'), arr(sub_i,'d_lab')
ok_ = np.isfinite(dn)&np.isfinite(dr)
print(f'head distance to: VIS label {np.median(dl[ok_]):.0f} | Rubin-mean center {np.median(dr[ok_]):.0f} '
      f'| NISP-mean center {np.median(dn[ok_]):.0f} mas  '
      f'(head closer to NISP than label: {(dn[ok_]<dl[ok_]).mean():.0%})')
plt.show()

# ---- scatter split: tight-head vs loose-head worseners ----
tight = sub_i[rms_hd[sub_i] < 15]; loose = sub_i[rms_hd[sub_i] >= 15]
t10_ = cKDTree(p10)
def pers(ii):
    d_,_ = t10_.query(np.column_stack([src_ra[ii]*cosd, src_dec[ii]]), k=1)
    return (d_*3600<0.5).mean()
print(f'\nscatter split of the {len(sub_i)} worseners (head cross-band RMS):')
for ii,tag in [(tight,'TIGHT (<15 mas)'),(loose,'loose (>=15 mas)')]:
    s_ = arr(ii,'sig')
    print(f'  {tag:16s}: N={len(ii):4d}  v10-persistent {pers(ii):5.0%}  '
          f'median k={np.median(k2[ii]):.0f}  chromatic sig>2: {(s_>2).mean():.0%}  '
          f'median head off {np.median([np.mean(RES[gid==i]) for i in ii]):.0f} mas')
print(f'  controls        : v10-persistent {pers(ctl_i):5.0%} (contamination reference)')
print(f'REFINED population: k>=3 material AND head RMS < 15 mas -> {len(tight)} sources')

In [ ]:
# ---- mechanism galleries: wavelength-colored centroids ----
import matplotlib.cm as cm
def draw_source2(ax, i, half_as=2.0, inset_as=0.5):
    t = pick_tile(i, half_as)
    if t is None: return False
    img, w = get_vis(t)
    x0, y0 = w.wcs_world2pix(mlra[i], mldec[i], 0)
    h = int(round(half_as/0.1))
    xi, yi = int(round(float(x0))), int(round(float(y0)))
    st = img[yi-h:yi+h, xi-h:xi+h]
    lo = np.nanmedian(st); sc = np.nanpercentile(st-lo, 99.5)
    st = np.where(np.isfinite(st), st, lo)
    ext = [xi-h-0.5, xi+h-0.5, yi-h-0.5, yi+h-0.5]
    m = gid==i
    cols = cm.turbo((LAM[BAND[m]]-0.35)/(1.8-0.35))
    cx, cy = w.wcs_world2pix(RA[m], DEC[m], 0)
    hx, hy = w.wcs_world2pix(np.mean(hd_ra[m]), np.mean(hd_dec[m]), 0)
    # fitted chromatic segment (blue end -> red end), if fit exists
    seg = None
    if i in CH and CH[i]['sig'] > 0:
        x = LAM[BAND[m]]; xc = x - x.mean(); dxy = -RAWV[m]
        sx = np.polyfit(xc, dxy[:,0], 1); sy = np.polyfit(xc, dxy[:,1], 1)
        pts=[]
        for xq in (x.min()-x.mean(), x.max()-x.mean()):
            off = np.array([np.polyval(sx,xq), np.polyval(sy,xq)])   # mas
            pra = mlra[i] + (off[0]/3.6e6)/cosd; pdec = mldec[i] + off[1]/3.6e6
            pts.append(w.wcs_world2pix(pra, pdec, 0))
        seg = np.array(pts, float)
    for a, zoom in [(ax, False), (ax.inset_axes([0.62,0.62,0.36,0.36]), True)]:
        a.imshow(np.arcsinh((st-lo)/max(sc/10,1e-9)), origin='lower', cmap='gray',
                 extent=ext, interpolation='nearest' if zoom else None)
        if seg is not None:
            a.plot(seg[:,0], seg[:,1], '-', c='magenta', lw=1.4 if zoom else 1.0, alpha=0.85)
        a.scatter(cx, cy, s=40 if zoom else 18, c=cols, lw=0.4, edgecolors='k')
        a.plot(hx, hy, 'x', c='#ff2222', ms=11 if zoom else 8, mew=2)
        a.plot(x0, y0, '+', c='white', ms=12 if zoom else 10, mew=1.7)
        if zoom:
            hz = inset_as/0.1
            a.set_xlim(x0-hz, x0+hz); a.set_ylim(y0-hz, y0+hz)
            for sp in a.spines.values(): sp.set_color('yellow')
        a.set_xticks([]); a.set_yticks([])
    ch = CH.get(i, {})
    ax.set_title(f'k={k2[i]}, S/N={src_snr[i]:.0f}, headRMS={rms_hd[i]:.0f}, '
                 f'$\\langle$off$\\rangle$={np.mean(RES[m]):.0f} mas, chrom sig={ch.get("sig",np.nan):.1f}',
                 fontsize=8.5)
    return True

sig_of = lambda i: CH[i]['sig'] if i in CH else 0.
cl_ok = lambda i: span[i] <= 0.3
grp_chrom = sorted([i for i in tight if cl_ok(i) and sig_of(i)>2], key=lambda i:-sig_of(i))[:12]
grp_achr  = sorted([i for i in tight if cl_ok(i) and sig_of(i)<=1],
                   key=lambda i: -(k2[i]*1e4 + w2res[i]))[:12]
grp_loose = sorted([i for i in loose if cl_ok(i)], key=lambda i:-rms_hd[i])[:8]
GALS = [('chromatic (tight head, sig>2)', grp_chrom, (3,4)),
        ('achromatic (tight head, sig<=1)', grp_achr, (3,4)),
        ('loose head RMS (discount candidates)', grp_loose, (2,4))]
for name, group, (nr,nc_) in GALS:
    fig, axs = plt.subplots(nr, nc_, figsize=(4*nc_, 4.25*nr))
    for ax, i in zip(np.ravel(axs), group):
        if not draw_source2(ax, i): ax.axis('off')
    for ax in np.ravel(axs)[len(group):]: ax.axis('off')
    fig.suptitle(f'{name} — dots = classical centroids colored by wavelength (turbo: blue u $\\to$ red H), '
                 f'magenta = fitted chromatic axis, red x = head mean, white + = VIS label',
                 fontsize=11.5, y=1.0)
    plt.tight_layout()
    fn = 'nb31_gal_' + name.split(' ')[0].strip('(') + '.png'
    fig.savefig(OUTDIR/fn, dpi=120, bbox_inches='tight')
    print('saved', (OUTDIR/fn).relative_to(REPO)); plt.show()

### Verdict: head vs classical (split by companionship)

First, a scale caveat the galleries make obvious: *everything* here is sub-pixel. Both estimators
put the source "at the center" to within half a VIS pixel; the disagreement is tens of mas. The
question is which one is closer to the light, and that is quantitative, not visual.

**Repeatability across bands** (all 604 worseners, split):

| group | classical RMS | head RMS | head tighter |
|---|---|---|---|
| worseners, isolated (N=461) | 31.1 mas | 13.0 mas | 96% |
| worseners, paired (N=143) | 41.6 mas | 22.1 mas | 92% |
| controls (N=604) | 39.2 mas | 9.9 mas | 99% |

The head is the low-noise estimator everywhere, so precision cannot decide; note it is less
self-consistent on worseners (13.0/22.1) than on controls (9.9) — it partially senses the trouble.

**Accuracy against the VIS light** (median distances, mas):

| group | → VIS peak | | → 0.75" flux centroid | |
|---|---|---|---|---|
| | head | label | head | label |
| worseners ISOLATED (197) | **50** | 23 | **39** | 15 |
| worseners paired (44) | **55** | 29 | **37** | 24 |
| controls (124) | 29 | 24 | 20 | 22 |

The isolated subgroup shows the *same* asymmetry as the paired one: the head sits 2–2.6× farther
from both the VIS peak and the flux-weighted centroid than the label does, while on controls head
and label are equivalent. So the pooled verdict was not carried by the blends. Classical band
centroids behave like the label (isolated: 34 / 19 mas).

**Not a matching artifact:** only 10 of 604 groups (2%) have label positions spanning > 0.5".

**Conclusion (calibrated):** on this ~1.6% population the head is *precise but biased*: it
reproduces a per-source, fixed ~40–50 mas offset in every band, and that offset does not
correspond to any natural photometric center we can find (not the peak, not the mean light, not
the multi-band classical mean) — for isolated and paired sources alike. The classical centroid is
noisier band-to-band but unbiased with respect to the light. All of this is at the sub-pixel
level: the head is not mis-detecting these galaxies, it is consistently off-center by a few tens
of mas on structured hosts. For science use, serve these sources the classical position or a
sigma-gated blend (the head's own sigma is elevated here), and describe the failure mode as a
reproducible morphology-driven bias.

### Mechanism summary (refined)

**Chromatic centers are real and common — but they don't define the population.** 46% of the
worseners have a significant (sig > 2) wavelength drift of their classical centroid… and so do 46%
of the S/N-matched controls (median |slope| 37 vs 44 mas/μm). Color-dependent centers are a
property of faint structured galaxies generally, not of the failure set.

**But when the head errs, it errs along the color axis.** Among chromatic worseners, the head's
offset lies along the fitted chromatic axis for 60% (|cos| > 0.7; random gives 49%), and it
prefers the **blue** end 43% vs 17% red. Consistently, the head's position is marginally closer to
the Rubin-band mean center (36 mas) than to the VIS label or the NISP mean (42 mas each). Reading:
on chromatic sources the head reports something closer to a blue/clump-weighted compromise center,
while the label is the VIS-band convention. (The +1 pile-up in the sig ≤ 1 control histogram is the
expected shared-label artifact; the −1 dominance at sig > 2 is the opposite sign, so not that
artifact.)

**The scatter split refines the population (as it should):**

| | N | v10-persistent | ⟨head off⟩ | chromatic sig>2 |
|---|---|---|---|---|
| tight (head RMS < 15 mas) | 315 | **89%** | 38 mas | 40% |
| loose (head RMS ≥ 15 mas) | 289 | 71% | 62 mas | 53% |
| controls | — | 1% | — | — |

- The **tight core (315 sources, 0.8% of the catalog)** is the real "model bias" population: one
  fixed wrong position, reproduced across bands and across independent trainings (89%).
- The **loose half** is *not* chance (71% persistence vs 1% contamination), but it is a different
  phenomenon: more chromatic, larger offsets, and the head wanders band-to-band just as the
  classical centroids do — these are galaxies whose center is genuinely ill-defined and
  color-dependent (the loose gallery shows clumpy/disturbed hosts with centroids marching along
  the structure). Calling the head "worse" there overstates the case: both estimators are
  measuring a moving target; these should be down-weighted rather than counted as head failures.

**Working picture:** the head has learned a morphology- and color-weighted center that
systematically disagrees with the VIS windowed-centroid convention on structured hosts — biased
toward the blue/clumpy light on chromatic sources, direction set by each galaxy's own structure
(hence isotropic across sources). The honest failure population for the paper is the tight core
(~315 sources); the loose half belongs in an "ill-defined chromatic centers" caveat, not in the
head's error budget.

## Fix candidates: biaspen and bandaware head retrains, judged on this testbench

Two single-change retrains of the head (2026-08-27, `train_latent_position_v2.py`, same data,
split, and recipe as the plain head; NO worsener ever used as a training signal, so this
population remains a fair test):
- **biaspen** — loss-only: two jitters per source, penalty `0.1 × mean(r₁·r₂)/(10 mas)²`
  (unbiased ‖bias‖² estimator, not σ-normalized). Targets the tight core.
- **bandaware** — architecture: wavelength embedding + auxiliary band-center output, band-centroid
  training rows from the anchors archive (train tiles only). Targets chromatic blindness.

Both evaluated with the same detector labels → anchors align row-for-row with the plain archive,
so the source grouping, S/N cells, and control sets carry over unchanged. Scoreboard: overall and
faint medians, coherent count (k≥3 material, ≥4 bands) vs its own chance null, tight core
(head RMS < 15 mas) and its overlap with the plain head's 315, and the controls' cross-band head
RMS (baseline 9.9 mas — degradation = bias traded for scatter).

Caveat carried forward: both variants trained with bottleneck window 11 (auto-scaled) vs the
plain head's 5, so variant-vs-plain includes that receptive-field difference; the two variants are
directly comparable to each other.

In [ ]:
# ---- three-head verdict table on the identical anchor set ----
VAR_ARCHS = {
 'biaspen':   REPO/'models/checkpoints/latent_position_v11_biaspen/anchors_centernet_v11biaspen.npz',
 'bandaware': REPO/'models/checkpoints/latent_position_v11_bandaware/anchors_centernet_v11bandaware.npz',
 'anchored':  REPO/'models/checkpoints/latent_position_v11_anchored/anchors_centernet_v11anchored.npz',
 'hinge':     REPO/'models/checkpoints/latent_position_v11_hinge/anchors_centernet_v11hinge.npz',
}
def swap_head_resid(archpath):
    d2 = np.load(archpath, allow_pickle=True)
    resv = np.vstack([np.asarray(d2[f'{b}_head_resid'])*1000. for b in BANDS])[ok][keep]
    ra2  = np.concatenate([np.asarray(d2[f'{b}_ra']) for b in BANDS])[ok][keep]
    assert np.allclose(ra2, RA, atol=1e-6), 'anchor rows do not align'
    return resv

plain_tight = set(tight.tolist())
def head_scoreboard(name, resv):
    res = np.hypot(*resv.T)
    w2 = (res > RAW + 10.) & (res > 20.)
    kk = np.bincount(gid, weights=w2, minlength=ncomp).astype(int)
    coh = np.flatnonzero((nb_src>=4)&(kk>=3))
    nul = null_sim(w2, nsims=100, seed=99)
    nul_c = nul[:,4:,3:].sum(axis=(1,2))
    hra = lab_ra - (resv[:,0]/3.6e6)/cosd; hdec = lab_dec - resv[:,1]/3.6e6
    rms_h = per_src_rms(hra, hdec)
    tt = coh[rms_h[coh] < 15]
    ovl = len(plain_tight & set(tt.tolist()))
    faint = SNR < 10
    print(f'{name:10s}| med {np.median(res):5.1f} faint-med {np.median(res[faint]):5.1f} mas | '
          f'coherent {len(coh):4d} (null {nul_c.mean():5.1f}) | tight core {len(tt):4d} '
          f'(overlap w/ plain 315: {ovl:3d}) | ctl headRMS {np.median(rms_h[ctl_i]):5.1f} mas')
    return dict(res=res, coh=coh, tight=tt, rms_h=rms_h)

print(f'{"head":10s}| overall + faint medians | coherent worseners | tight core | control scatter')
res_plain = head_scoreboard('plain', RESV)
SB = {n: head_scoreboard(n, swap_head_resid(p_)) for n,p_ in VAR_ARCHS.items()}

# per-source improvement on the plain tight core: head offset before vs after
fig,axs = plt.subplots(1,2,figsize=(13.5,5))
tt_i = np.array(sorted(plain_tight))
for name,col in [('biaspen','#3b6fb6'),('bandaware','#c84b4b'),
                 ('anchored','#2e8b57'),('hinge','#a030d0')]:
    off_new = np.array([np.mean(SB[name]['res'][gid==i]) for i in tt_i])
    off_old = np.array([np.mean(RES[gid==i]) for i in tt_i])
    axs[0].scatter(off_old, off_new, s=8, alpha=0.5, color=col, label=name)
    axs[1].hist(off_new-off_old, bins=np.linspace(-80,80,45), histtype='step', lw=2,
                color=col, label=f'{name}: med change {np.median(off_new-off_old):+.0f} mas')
mx=120; axs[0].plot([0,mx],[0,mx],'k--',lw=1)
axs[0].set(xlabel='plain head: mean offset on tight core [mas]',
           ylabel='variant head: mean offset [mas]', xlim=(0,mx), ylim=(0,mx))
axs[0].legend(); axs[1].axvline(0,color='k',lw=1)
axs[1].set(xlabel='change in mean offset on the plain 315 tight-core sources [mas]')
axs[1].legend(fontsize=9)
plt.tight_layout()
fig.savefig(OUTDIR/'nb31_fix_verdict.png',dpi=130,bbox_inches='tight')
print('saved',(OUTDIR/'nb31_fix_verdict.png').relative_to(REPO)); plt.show()

### Fix verdict (final, four variants)

| head | overall med | faint med | coherent (null) | tight core | ∩ plain 315 | ctl RMS |
|---|---|---|---|---|---|---|
| plain | 14.4 | 18.8 | 604 (22) | 315 | — | 9.9 |
| biaspen | 15.1 | 19.7 | 542 (18) | 297 | 260 | 10.5 |
| bandaware | 9.7 | 10.9 | 1113 (89) | 662 | 283 | 6.8 |
| **anchored** | **7.2** | **8.6** | **235 (2)** | **37** | **6** | 7.7 |
| hinge | 9.6 | 10.8 | 1061 (82) | 639 | 282 | 6.9 |

(deduped-measurement basis; on the all-anchors basis the eval medians are anchored 9.2 / hinge 6.8 /
bandaware 9.2 — orderings on the bright-weighted dedup basis differ, comparisons within a basis are
exact since all heads score identical rows)

**anchored wins on every axis at once.** Best overall and faint medians (−50% and −54% vs plain,
beating bandaware too), controls' precision kept (7.7 mas), and — the point of the design — **the
tight core collapses from 315 to 37 sources, with only 6 of the original 315 surviving**. The
coherent population shrinks 604 → 235 (its ×107 excess over the null says a residual, much smaller
failure set remains — plausibly faint sources where the anchor is noisy and the bound is open).
Building the label convention into the forward pass did what no training-signal change could:
the architecture now *computes* the windowed centroid instead of having to imitate it.

**hinge: global ≈ bandaware, tail ≈ bandaware — the penalty did not deliver.** It engaged during
training (frac_hinged 9.6% → 4.3%, loss halved) but at eval the coherent population (1061) and
tight core (639) match bandaware's. Reading: the hinge reduced *training-row* violations — which
are dominated by easy cases — while the tight-core sources, whose features simply do not encode
the convention centroid, remained unfixable by gradient pressure at any loss shape. Third
loss-based fix to fail on this population (NLL, biaspen, hinge): consistent with the failure being
representational, not an optimization artifact.

**Recommendation:** adopt the anchored head as the production astrometry head, pending the two
standard gates: patch-disjoint (patchval25-style) validation and an EDF-S OOD check. Remaining
follow-ups: characterize the residual 37/235 population (prediction: faint, low anchor-S/N), and
re-run the router/sigma calibration on the anchored head's sigma.

*Teacher caveat (carried from the discussion above):* anchored is the most convention-loyal design
possible — it perfects "amortized classical" and inherits the convention's biases by construction.
Beating the convention itself would require the joint multi-band canonical labels, injection-truth
training, or repeatability objectives.

**Adoption gates (2026-08-31, both PASSED):**
- **EDF-S OOD** (72 tiles, no retraining, plain-head recipe): raw 59.1/46.1 → anchored MAE 15.8,
  median 6.4 mas vs plain's 27.3/13.9 — the OOD error is halved and matches in-field performance;
  the anchor computes on pixels, so the robustness is structural.
- **Patch-disjoint** (retrained with patch 25 held out, evaluated on the 108 patch-25 tiles,
  82,182 anchors): raw 60.5/48.0 → MAE 16.1, median 6.9 mas ≈ the random-split result (18.9/9.2)
  — no tile-overlap leakage behind the gains.

Anchored head checkpoints: `latent_position_v11_anchored` (production candidate),
`latent_position_v11_anchored_patchval25` (gate), EDF-S anchors `anchors_edfs_v11anchored.npz`.

## Full-population check: raw vs head offset, plain v11 vs anchored

The fix analysis above focused on the failure tail. This section is the guard against the opposite
failure mode: a head that "fixes the 315" by hand while quietly degrading everyone else. All
measurements (deduped basis, identical rows for both heads) go into a raw-vs-head hexbin per head:
if the anchored gains are real and global, the mass must sit below the 1:1 line **everywhere**,
the running median must drop at every raw-offset level, the median-vs-S/N curve must improve at
every S/N (not just where the worseners live), and the worsened fraction must fall in every raw
bin — not just in the material-worsening corner.

Plot notes: hex cells with fewer than 10 measurements are suppressed (`mincnt=10`) so the
panels show populations, not single noisy measurements. The hard edge at raw = 200 mas is the
eval's own pre-selection (`--clip-mas 200`: raw offsets beyond that are rejected as bad
centroids / wrong matches before any head runs), applied identically to both heads.
"Material" worsening repeats the W2 definition from the coherence section: the head is at
least 10 mas worse than classical AND lands more than 20 mas off — i.e. it screens out
inconsequential worsenings of already-good positions.
Axes run to 0.1 mas with no clipping, so the sub-mas tail is shown as it is. A sub-mas "offset"
against a label with a 5–16 mas noise floor is not sub-mas accuracy: it is the head's estimate
and the label happening to coincide, and the smooth rise of counts toward small offsets is
exactly the chance-coincidence (Rayleigh, density ∝ r) prediction — plain 1.4% vs anchored 3.8%
below 1 mas simply reflects the anchored head's tighter scatter around the label, not a
distinct population.

In [ ]:
# ---- full-population raw vs head hexbins + conditional medians ----
res_anch = SB['anchored']['res']
w_pl = RES > RAW; w_an = res_anch > RAW
m_pl = (RES > RAW+10.)&(RES > 20.); m_an = (res_anch > RAW+10.)&(res_anch > 20.)

fig, axs = plt.subplots(2, 2, figsize=(13.5, 11))
lo, hi = 0.1, 500.                      # log-log axes (nb18 style); no clipping
for ax, res_, name in [(axs[0,0], RES, 'plain v11 head'), (axs[0,1], res_anch, 'anchored head')]:
    hb = ax.hexbin(RAW, res_, gridsize=70, bins='log', xscale='log', yscale='log',
                   extent=(np.log10(lo), np.log10(hi), np.log10(lo), np.log10(hi)),
                   cmap='viridis', mincnt=10)
    ax.plot([lo,hi],[lo,hi],'r--',lw=1.2,label='1:1 (head = classical)')
    qb = np.geomspace(2., hi, 25)
    mid = np.sqrt(qb[1:]*qb[:-1])
    med = [np.median(res_[(RAW>=a)&(RAW<b)]) if ((RAW>=a)&(RAW<b)).sum()>30 else np.nan
           for a,b in zip(qb[:-1],qb[1:])]
    ax.plot(mid, med, 'w-', lw=3); ax.plot(mid, med, 'k-', lw=1.4, label='running median')
    ax.set(xlim=(lo,hi), ylim=(lo,hi), xlabel='raw offset (classical band centroid) [mas]',
           ylabel='head offset [mas]')
    fw = 100*np.mean(res_>RAW); fm = 100*np.mean((res_>RAW+10.)&(res_>20.))
    ax.set_title(f'{name}: med {np.median(res_):.1f} mas | worsened {fw:.1f}% | material {fm:.2f}%')
    ax.legend(loc='upper left', fontsize=9); plt.colorbar(hb, ax=ax, label='log N')

sq = np.percentile(SNR, np.linspace(0,100,13)); sq[0]-=1
smid = 0.5*(sq[1:]+sq[:-1])
for arr,name,col,ls in [(RAW,'raw (classical)','0.45','--'),(RES,'plain head','#3b6fb6','-'),
                        (res_anch,'anchored head','#2e8b57','-')]:
    m = [np.median(arr[(SNR>a)&(SNR<=b)]) for a,b in zip(sq[:-1],sq[1:])]
    axs[1,0].plot(smid, m, 'o'+ls, color=col, label=name)
axs[1,0].set(xscale='log', xlabel='S/N (per-band)', ylabel='median offset [mas]',
             title='improvement at every S/N, not only where worseners live')
axs[1,0].legend(); axs[1,0].grid(alpha=0.3)

rb = np.percentile(RAW, np.linspace(0,100,13)); rb[0]=0.
rmid = 0.5*(rb[1:]+rb[:-1])
for wv, mv, name, col in [(w_pl,m_pl,'plain','#3b6fb6'),(w_an,m_an,'anchored','#2e8b57')]:
    fw = [100*np.mean(wv[(RAW>=a)&(RAW<b)]) for a,b in zip(rb[:-1],rb[1:])]
    fm = [100*np.mean(mv[(RAW>=a)&(RAW<b)]) for a,b in zip(rb[:-1],rb[1:])]
    axs[1,1].plot(rmid, fw, 'o-', color=col, label=f'{name}: worsened (res>raw)')
    axs[1,1].plot(rmid, fm, 's--', color=col, label=f'{name}: material')
axs[1,1].set(xlabel='raw offset [mas]', ylabel='% of measurements', yscale='log',
             title='worsened fraction per raw-offset bin')
axs[1,1].legend(fontsize=9); axs[1,1].grid(alpha=0.3)
plt.tight_layout()
fig.savefig(OUTDIR/'nb31_full_population_plain_vs_anchored.png', dpi=130, bbox_inches='tight')
print('saved', (OUTDIR/'nb31_full_population_plain_vs_anchored.png').relative_to(REPO)); plt.show()

# quantile table over the WHOLE population + per-S/N-tercile medians
print('\npercentiles of offset [mas] over ALL deduped measurements:')
pcts = [25,50,75,90,95,99]
print(f'{"":12s}' + ''.join(f'p{p:<7d}' for p in pcts))
for arr,name in [(RAW,'raw'),(RES,'plain'),(res_anch,'anchored')]:
    print(f'{name:12s}' + ''.join(f'{np.percentile(arr,p):<8.1f}' for p in pcts))
t1,t2 = np.percentile(SNR,[33.3,66.7])
print('\nmedian offset by S/N tercile (faint | mid | bright):')
for arr,name in [(RAW,'raw'),(RES,'plain'),(res_anch,'anchored')]:
    print(f'{name:12s}' + ' | '.join(f'{np.median(arr[m]):6.1f}'
          for m in [SNR<=t1,(SNR>t1)&(SNR<=t2),SNR>t2]))
print(f'\nimproved vs raw: plain {100*np.mean(RES<RAW):.1f}%  anchored {100*np.mean(res_anch<RAW):.1f}%')
print(f'anchored better than plain (per measurement): {100*np.mean(res_anch<RES):.1f}%')

### Reading: the gains are global, not tail surgery

- **Hexbins (log–log, nb18 style):** the anchored cloud sits below the plain cloud at *every*
  raw offset; the running median is lower across the full 2–500 mas range. No raw regime got
  worse to pay for the tail fix.
- **Percentiles:** anchored beats plain at every quantile — p25 3.3 vs 5.7, p50 7.2 vs 14.4,
  p90 44.7 vs 79.4, p99 128 vs 160 mas. A hand-fix of 315 sources (0.8%) could not move p25.
- **vs S/N:** improvement in every S/N bin — faint tercile 19.7 → 8.5, bright 9.4 → 5.3 mas.
  The faint end (where no worseners were flagged) improves the *most*, because that is where the
  open bound lets the learned residual work.
- **Worsened fractions:** lower in every raw-offset bin, for both the any-worsening and the
  material definitions (overall 8.9% → 4.6% and 3.20% → 1.79%). Anchored is better than plain on
  78% of individual measurements and improves on raw for 95.4% (plain: 91.1%).

Conclusion: the anchor is a uniform change to the estimator, not a patch on the failure list —
consistent with its construction (no source is special-cased; the bound depends only on local
pixel S/N).